In [ ]:
import pandas as pd

df = pd.read_parquet("../../data/train_data/final_dataset.parquet")


In [ ]:
df.columns


In [ ]:
df['classLabel'].value_counts()


In [ ]:
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

def make_train_val_from_llmtrace(
    *,
    split: str = "train",
    lang: str = "ru",
    dataset_name: str = "iitolstykh/LLMTrace_classification",
    val_size: float = 0.1,
    seed: int = 40,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    ds = load_dataset(dataset_name, split=split)

    if lang is not None:
        ds = ds.filter(lambda x: x["lang"] == lang)

    df = ds.to_pandas()
    df["y"] = df["label"].astype(str).str.lower().map({"ai": 1, "human": 0})
    df["text_for_clf"] = df["text"].astype(str)


    df = df[df["y"].isin([0, 1])]
    df = df[df["text_for_clf"].str.strip().str.len() > 0].reset_index(drop=True)

    df_train, df_val = train_test_split(
        df,
        test_size=val_size,
        random_state=seed,
        stratify=df["y"],
    )

    return df_train.reset_index(drop=True), df_val.reset_index(drop=True), df


In [ ]:
import pandas as pd

def make_test_from_final_parquet(
    parquet_path: str,
    *,
    seed: int = 40,
    balance: bool = True,
) -> pd.DataFrame:
    df = pd.read_parquet(parquet_path)

    df_ai = df[df["classLabel"].isin(["AI", "human+AI"])].copy()
    df_hu = df[df["classLabel"].astype(str) == "human"].copy()

    df_ai["text_for_clf"] = df_ai["generatedText"].astype(str)
    df_hu["text_for_clf"] = df_hu["text"].astype(str)

    df_ai = df_ai[df_ai["text_for_clf"].str.strip().str.len() > 0]
    df_hu = df_hu[df_hu["text_for_clf"].str.strip().str.len() > 0]

    df_ai["y"] = 1
    df_hu["y"] = 0

    if balance:
        n = min(len(df_ai), len(df_hu))
        df_ai = df_ai.sample(n, random_state=seed)
        df_hu = df_hu.sample(n, random_state=seed)

    df_test = pd.concat([df_ai, df_hu], ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return df_test


In [ ]:
df_ai = df[df["classLabel"].isin(["AI", "human+AI"])].copy()
df_hu = df[df["classLabel"].astype(str) == "human"].copy()

df_ai["text_for_clf"] = df_ai["generatedText"].astype(str)

df_hu["text_for_clf"] = df_hu["text"].astype(str)

df_ai = df_ai[df_ai["text_for_clf"].str.strip().str.len() > 0]
df_hu = df_hu[df_hu["text_for_clf"].str.strip().str.len() > 0]

df_ai["y"] = 1
df_hu["y"] = 0

n = min(len(df_ai), len(df_hu))
df_train_all = pd.concat([df_ai, df_hu], ignore_index=True).sample(frac=1.0, random_state=40).reset_index(drop=True)

print(df_train_all[["y"]].value_counts())

x_column = 'text_for_clf'
target_column = 'y'


In [ ]:



from typing import List, Dict, Union
import torch
from transformers import AutoTokenizer

def make_tokenize_fn(tokenizer, max_length: int = 256):

    def tokenize(texts: Union[str, List[str]]) -> Dict[str, torch.Tensor]:
        if isinstance(texts, str):
            texts_ = [texts]
        else:
            texts_ = list(texts)

        enc = tokenizer(
            texts_,
            truncation=True,
            max_length=max_length,
            padding=True,
            return_tensors="pt",
        )
        return enc
    return tokenize


In [ ]:
import json
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm.auto import tqdm


class TextClfDataset(Dataset):
    def __init__(self, texts: List[str], labels: List[int]):
        self.texts = list(texts)
        self.labels = list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return {"text": self.texts[i], "label": int(self.labels[i])}


def train_simple_rubert_tiny2(
    df_train: pd.DataFrame,
    df_val: pd.DataFrame,
    *,
    text_col: str = "text_for_clf",
    label_col: str = "y",
    model_name: str = "cointegrated/rubert-tiny2",
    max_length: int = 256,
    batch_size: int = 32,
    lr: float = 2e-5,
    weight_decay: float = 0.01,
    epochs: int = 3,
    seed: int = 40,
    device: str = None,
    save_dir: str = "models",
):

    torch.manual_seed(seed)
    np.random.seed(seed)

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    os.makedirs(save_dir, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenize = make_tokenize_fn(tokenizer, max_length=max_length)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model.to(device)


    train_ds = TextClfDataset(df_train[text_col].astype(str).tolist(), df_train[label_col].astype(int).tolist())
    val_ds   = TextClfDataset(df_val[text_col].astype(str).tolist(),   df_val[label_col].astype(int).tolist())


    def collate_fn(batch):
        texts = [b["text"] for b in batch]
        labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
        enc = tokenize(texts)
        enc["labels"] = labels
        return enc

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size * 2, shuffle=False, collate_fn=collate_fn)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    def eval_on_val():
        model.eval()
        all_preds = []
        all_labels = []

        total_loss = 0.0
        n_steps = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc="val", leave=False):
                labels = batch.pop("labels").to(device)
                batch = {k: v.to(device) for k, v in batch.items()}

                out = model(**batch, labels=labels)
                loss = out.loss
                logits = out.logits

                total_loss += float(loss.item())
                n_steps += 1

                preds = torch.argmax(logits, dim=-1).detach().cpu().numpy()
                all_preds.append(preds)
                all_labels.append(labels.detach().cpu().numpy())

        y_true = np.concatenate(all_labels)
        y_pred = np.concatenate(all_preds)

        acc = accuracy_score(y_true, y_pred)


        p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
            y_true, y_pred, average="macro", zero_division=0
        )

        p_ai, r_ai, f_ai, _ = precision_recall_fscore_support(
            y_true, y_pred, labels=[1], average=None, zero_division=0
        )
        p_ai = float(p_ai[0]); r_ai = float(r_ai[0]); f_ai = float(f_ai[0])

        return {
            "val_loss": total_loss / max(1, n_steps),
            "accuracy": acc,
            "precision_macro": float(p_macro),
            "recall_macro": float(r_macro),
            "f1_macro": float(f_macro),
            "precision_AI": p_ai,
            "recall_AI": r_ai,
            "f1_AI": f_ai,
        }


    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        pbar = tqdm(train_loader, desc=f"train epoch {epoch}/{epochs}")
        running_loss = 0.0
        n_steps = 0

        for batch in pbar:
            labels = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad(set_to_none=True)

            out = model(**batch, labels=labels)
            loss = out.loss
            loss.backward()

            optimizer.step()

            running_loss += float(loss.item())
            n_steps += 1
            pbar.set_postfix(loss=running_loss / n_steps)


        metrics = eval_on_val()
        metrics["epoch"] = epoch
        metrics["train_loss"] = running_loss / max(1, n_steps)
        history.append(metrics)

        print(
            f"[epoch {epoch}] "
            f"train_loss={metrics['train_loss']:.4f} "
            f"val_loss={metrics['val_loss']:.4f} "
            f"acc={metrics['accuracy']:.4f} "
            f"P_AI={metrics['precision_AI']:.4f} "
            f"R_AI={metrics['recall_AI']:.4f}"
        )


        epoch_dir = os.path.join(save_dir, f"epoch_{epoch:02d}")
        os.makedirs(epoch_dir, exist_ok=True)


        model.save_pretrained(epoch_dir)

        tokenizer.save_pretrained(epoch_dir)


        with open(os.path.join(epoch_dir, "metrics.json"), "w", encoding="utf-8") as f:
            json.dump(metrics, f, ensure_ascii=False, indent=2)

    return model, tokenizer, history


In [ ]:
df_train_llmtrace, df_val_llmtrace, df_llmtrace = make_train_val_from_llmtrace(
    split='train',
    lang="ru",
    val_size=0.1,
    seed=40,
)



In [ ]:
print(df_train_llmtrace['data_type'].value_counts())
df_train_llmtrace.columns


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
import time
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification


def predict_text(
    text: str,
    *,
    model_dir: str,
    max_length: int = 256,
    device: str = None,
):
    t0 = time.perf_counter()

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"


    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()


    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        padding=False,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}


    with torch.no_grad():
        out = model(**enc)
        logits = out.logits.squeeze(0)

    probs = F.softmax(logits, dim=-1)
    p_not_ai = float(probs[0].item())
    p_ai = float(probs[1].item())

    label = "AI" if p_ai >= 0.5 else "not_ai"

    elapsed = time.perf_counter() - t0

    return {
        "label": label,
        "proba": {
            "AI": p_ai,
            "not_ai": p_not_ai,
        },
        "logits": [float(logits[0].item()), float(logits[1].item())],
        "time_sec": elapsed,
    }


In [ ]:
text = """Период с 1922 по 1939 год — время становления советского государства, модернизации экономики страны, а также признания СССР на международной арене.
Приход в 1924 году к власти И. В. Сталина ознаменовал переход к новому этапу реформ во всех сферах жизни общества.
В данный период произошло много событий: продолжение реализации новой экономической политики — вплоть до 1928 года, образование СССР — 30 декабря 1922 года, принятие Конституции – 1924, 1936 гг., проведение политики индустриализации, первые пятилетние планы — 1928 - 1933, 1933 - 1937 гг., проведение"""

res = predict_text(
    text,
    model_dir="../../trained_models/models_DEEP_PAVLOV_llm_trace/epoch_03",
    device='cpu'
)

res


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)
from tqdm.auto import tqdm


def evaluate_on_llmtrace(
    *,
    model_dir: str,
    split: str = "test",
    lang: str = "ru",
    batch_size: int = 64,
    max_length: int = 256,
    device: str = None,
    dataset_name: str = "iitolstykh/LLMTrace_classification",
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"


    ds = load_dataset(dataset_name, split=split)


    if lang is not None:
        ds = ds.filter(lambda x: x["lang"] == lang)


    def to_y_true(lbl: str) -> int:

        lbl = (lbl or "").strip().lower()
        if lbl == "ai":
            return 1
        if lbl == "human":
            return 0
        raise ValueError(f"Unknown label in dataset: {lbl!r}")


    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()

    y_true = []
    y_pred = []


    for i in tqdm(range(0, len(ds), batch_size), desc=f"LLMTrace eval ({split}, {lang})"):
        batch = ds[i : i + batch_size]

        texts = batch["text"]
        labels = batch["label"]

        enc = tokenizer(
            texts,
            truncation=True,
            max_length=max_length,
            padding=True,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            logits = model(**enc).logits
            preds = torch.argmax(logits, dim=-1).detach().cpu().numpy()

        y_pred.append(preds)
        y_true.append(np.array([to_y_true(x) for x in labels], dtype=np.int64))

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)


    acc = accuracy_score(y_true, y_pred)

    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )


    p_ai, r_ai, f_ai, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], average=None, zero_division=0
    )
    p_ai, r_ai, f_ai = float(p_ai[0]), float(r_ai[0]), float(f_ai[0])


    p_na, r_na, f_na, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0], average=None, zero_division=0
    )
    p_na, r_na, f_na = float(p_na[0]), float(r_na[0]), float(f_na[0])

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    report = classification_report(
        y_true, y_pred,
        labels=[0, 1],
        target_names=["not_ai(human)", "AI(ai)"],
        digits=4,
        zero_division=0,
    )

    results = {
        "split": split,
        "lang": lang,
        "n": int(len(y_true)),
        "accuracy": float(acc),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_macro": float(f_macro),
        "precision_AI": p_ai,
        "recall_AI": r_ai,
        "f1_AI": f_ai,
        "precision_not_ai": p_na,
        "recall_not_ai": r_na,
        "f1_not_ai": f_na,
        "confusion_matrix_(rows=true_[not_ai,AI])_(cols=pred_[not_ai,AI])": cm,
        "classification_report": report,
    }

    print("\n=== RESULTS ===")
    print(f"dataset: {dataset_name} | split={split} | lang={lang} | n={results['n']}")
    print(f"accuracy        : {results['accuracy']:.4f}")
    print(f"macro P/R/F1    : {results['precision_macro']:.4f} / {results['recall_macro']:.4f} / {results['f1_macro']:.4f}")
    print(f"AI    P/R/F1    : {results['precision_AI']:.4f} / {results['recall_AI']:.4f} / {results['f1_AI']:.4f}")
    print(f"notAI P/R/F1    : {results['precision_not_ai']:.4f} / {results['recall_not_ai']:.4f} / {results['f1_not_ai']:.4f}")
    print("\nconfusion matrix (true rows, pred cols) [not_ai, AI]:")
    print(results["confusion_matrix_(rows=true_[not_ai,AI])_(cols=pred_[not_ai,AI])"])
    print("\nclassification report:")
    print(results["classification_report"])

    return results


In [ ]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from tqdm.auto import tqdm

def evaluate_on_dataframe(
    *,
    model_dir: str,
    df_test: pd.DataFrame,
    text_col: str = "text_for_clf",
    label_col: str = "y",
    batch_size: int = 64,
    max_length: int = 256,
    device: str = None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()

    texts = df_test[text_col].astype(str).tolist()
    labels = df_test[label_col].astype(int).tolist()

    y_true = []
    y_pred = []

    for i in tqdm(range(0, len(texts), batch_size), desc="eval df"):
        batch_texts = texts[i:i+batch_size]
        batch_labels = labels[i:i+batch_size]

        enc = tokenizer(
            batch_texts,
            truncation=True,
            max_length=max_length,
            padding=True,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            logits = model(**enc).logits
            preds = torch.argmax(logits, dim=-1).detach().cpu().numpy()

        y_pred.append(preds)
        y_true.append(np.array(batch_labels, dtype=np.int64))

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    acc = accuracy_score(y_true, y_pred)

    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )


    p_ai, r_ai, f_ai, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], average=None, zero_division=0
    )
    p_ai, r_ai, f_ai = float(p_ai[0]), float(r_ai[0]), float(f_ai[0])


    p_na, r_na, f_na, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0], average=None, zero_division=0
    )
    p_na, r_na, f_na = float(p_na[0]), float(r_na[0]), float(f_na[0])

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    report = classification_report(
        y_true, y_pred,
        labels=[0, 1],
        target_names=["not_ai(human)", "AI(ai)"],
        digits=4,
        zero_division=0,
    )

    print("\n=== TEST RESULTS (final_dataset.parquet) ===")
    print(f"n={len(y_true)}")
    print(f"accuracy        : {acc:.4f}")
    print(f"macro P/R/F1    : {p_macro:.4f} / {r_macro:.4f} / {f_macro:.4f}")
    print(f"AI    P/R/F1    : {p_ai:.4f} / {r_ai:.4f} / {f_ai:.4f}")
    print(f"notAI P/R/F1    : {p_na:.4f} / {r_na:.4f} / {f_na:.4f}")
    print("\nconfusion matrix (true rows, pred cols) [not_ai, AI]:")
    print(cm)
    print("\nclassification report:")
    print(report)

    return {
        "n": int(len(y_true)),
        "accuracy": float(acc),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_macro": float(f_macro),
        "precision_AI": float(p_ai),
        "recall_AI": float(r_ai),
        "f1_AI": float(f_ai),
        "precision_not_ai": float(p_na),
        "recall_not_ai": float(r_na),
        "f1_not_ai": float(f_na),
        "confusion_matrix": cm,
        "classification_report": report,
    }


In [ ]:
df_test_final = make_test_from_final_parquet(
    "scripts/data/final_dataset.parquet",
    seed=40,
    balance=True,
)

res_test = evaluate_on_dataframe(
    model_dir="models_llm_trace/epoch_03",
    df_test=df_test_final,
    text_col="text_for_clf",
    label_col="y",
    batch_size=64,
    max_length=256,
)


In [ ]:
import os
import json
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)
from tqdm.auto import tqdm


def _safe_read_json_file(path: str) -> Optional[Any]:

    try:
        with open(path, "r", encoding="utf-8") as f:
            txt = f.read().strip()
        if not txt:
            return None


        if "\n" in txt and txt.lstrip().startswith("{") and not txt.lstrip().startswith("["):
            objs = []
            for line in txt.splitlines():
                line = line.strip()
                if not line:
                    continue
                try:
                    objs.append(json.loads(line))
                except Exception:

                    continue
            return objs if objs else None


        return json.loads(txt)
    except Exception:
        return None


def load_ai_json_folder_to_df(
    folder: str,
    *,
    text_key: str = "text",
    label_key: str = "source",


    ai_labels: Tuple[str, ...] = ("ai", "ai+rew", "ai+gen", "generated", "machine"),
    human_labels: Tuple[str, ...] = ("human", "orig", "original", "not_ai"),
) -> pd.DataFrame:

    rows: List[Dict[str, Any]] = []
    paths = []

    for name in os.listdir(folder):
        if name.lower().endswith((".json", ".jsonl")):
            paths.append(os.path.join(folder, name))

    paths.sort()

    for p in tqdm(paths, desc="read json files"):
        parsed = _safe_read_json_file(p)
        if parsed is None:
            continue

        objs = parsed if isinstance(parsed, list) else [parsed]

        for obj in objs:
            if not isinstance(obj, dict):
                continue

            text = obj.get(text_key, None)
            if text is None:
                continue
            text = str(text).strip()
            if not text:
                continue

            raw_label = obj.get(label_key, None)
            raw_label_s = str(raw_label).strip().lower() if raw_label is not None else ""


            y = None
            if raw_label_s in [x.lower() for x in ai_labels]:
                y = 1
            elif raw_label_s in [x.lower() for x in human_labels]:
                y = 0
            else:

                y = None

            row = {
                "text_for_clf": text,
                "y": y,
                "rawLabel": raw_label,
                "jsonFile": os.path.basename(p),
            }


            for k in ["id", "model", "paraphrasing_type", "dataset", "source"]:
                if k in obj:
                    row[k] = obj.get(k)

            rows.append(row)

    df = pd.DataFrame(rows)


    if not df.empty:
        df["text_for_clf"] = df["text_for_clf"].astype(str)
        df = df[df["text_for_clf"].str.len() > 0].reset_index(drop=True)

    return df


def print_class_distribution(df: pd.DataFrame, label_col: str = "y"):
    if df.empty:
        print("DataFrame is empty.")
        return
    print("\nClass distribution (including NaN):")
    print(df[label_col].value_counts(dropna=False))
    print("\nClass distribution (valid only):")
    print(df[df[label_col].isin([0, 1])][label_col].value_counts())


def evaluate_model_on_df(
    *,
    model_dir: str,
    df: pd.DataFrame,
    text_col: str = "text_for_clf",
    label_col: str = "y",
    batch_size: int = 64,
    max_length: int = 256,
    threshold: float = 0.5,
    device: str = None,
):
    """
    Evaluates saved HuggingFace model on a DataFrame.
    Assumes labels: 0=not_ai, 1=AI
    Uses threshold on P(AI).
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    df_eval = df[df[label_col].isin([0, 1])].copy()
    if df_eval.empty:
        raise ValueError("No valid labeled rows found (y must be 0/1).")

    texts = df_eval[text_col].astype(str).tolist()
    y_true = df_eval[label_col].astype(int).to_numpy()

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()

    probs_all = []

    for i in tqdm(range(0, len(texts), batch_size), desc="model inference"):
        batch_texts = texts[i:i + batch_size]

        enc = tokenizer(
            batch_texts,
            truncation=True,
            max_length=max_length,
            padding=True,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1)
            probs_ai = probs[:, 1]

        probs_all.append(probs_ai.cpu().numpy())

    y_prob = np.concatenate(probs_all)
    y_pred = (y_prob >= threshold).astype(int)


    acc = accuracy_score(y_true, y_pred)

    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    p0, r0, f0, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0], average=None, zero_division=0
    )
    p1, r1, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], average=None, zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    report = classification_report(
        y_true, y_pred,
        labels=[0, 1],
        target_names=["not_ai", "AI"],
        digits=4,
        zero_division=0,
    )

    print("\n=== RESULTS ===")
    print(f"n={len(y_true)}")
    print(f"threshold       : {threshold:.3f}")
    print(f"accuracy        : {acc:.4f}")
    print(f"macro P/R/F1    : {p_macro:.4f} / {r_macro:.4f} / {f_macro:.4f}")
    print(f"not_ai P/R/F1   : {p0[0]:.4f} / {r0[0]:.4f} / {f0[0]:.4f}")
    print(f"AI     P/R/F1   : {p1[0]:.4f} / {r1[0]:.4f} / {f1[0]:.4f}")

    print("\nconfusion matrix (true rows, pred cols) [not_ai, AI]:")
    print(cm)

    print("\nclassification report:")
    print(report)

    return {
        "n": int(len(y_true)),
        "threshold": float(threshold),
        "accuracy": float(acc),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_macro": float(f_macro),
        "precision_not_ai": float(p0[0]),
        "recall_not_ai": float(r0[0]),
        "f1_not_ai": float(f0[0]),
        "precision_AI": float(p1[0]),
        "recall_AI": float(r1[0]),
        "f1_AI": float(f1[0]),
        "confusion_matrix": cm,
        "classification_report": report,
    }


def test_model_on_json_folder(
    *,
    json_folder: str,
    model_dir: str,
    text_key: str = "text",
    label_key: str = "source",
    batch_size: int = 64,
    max_length: int = 256,
    threshold = 0.5,
):
    df = load_ai_json_folder_to_df(
        json_folder,
        text_key=text_key,
        label_key=label_key,
        ai_labels=("ai", "ai+rew", "ai+gen"),
        human_labels=("human", "orig", "original", "not_ai"),
    )

    print(f"\nLoaded rows: {len(df)}")
    print_class_distribution(df, "y")


    res = evaluate_model_on_df(
        model_dir=model_dir,
        df=df,
        batch_size=batch_size,
        max_length=max_length,
        threshold=threshold
    )

    return df, res


In [ ]:
df_test, metrics = test_model_on_json_folder(
    json_folder="scripts/Ru-hard-detection-dataset-main/long_sc",
    model_dir="models_llm_trace/epoch_03",
    text_key="text",
    label_key="source",
    batch_size=64,
    max_length=256,
    threshold=0.99
)


In [ ]:
import os
import json
import numpy as np
import pandas as pd
from typing import List, Tuple, Optional

from sklearn.model_selection import train_test_split


import numpy as np
import pandas as pd
from typing import List, Tuple
from sklearn.model_selection import train_test_split


def build_binary_dataset_for_generators(
    df: pd.DataFrame,
    *,
    generators: List[str],
    text_col: str = "text_for_clf",
    y_col: str = "y",
    ai_model_col: str = "model",
    val_size: float = 0.1,
    seed: int = 40,

    max_ai: int | None = None,
    shuffle: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Требование:
      - нолики: берем ВСЕ доступные (y==0)
      - единички: берем только из generators (y==1 and model in generators)
      - опционально можно ограничить число единичек параметром max_ai (иначе берем все)
    """

    df = df.copy()


    df_ai = df[(df[y_col] == 1) & (df[ai_model_col].astype(str).isin(generators))].copy()
    df_hu = df[df[y_col] == 0].copy()

    df_ai[text_col] = df_ai[text_col].astype(str)
    df_hu[text_col] = df_hu[text_col].astype(str)

    df_ai = df_ai[df_ai[text_col].str.strip().str.len() > 0]
    df_hu = df_hu[df_hu[text_col].str.strip().str.len() > 0]

    if len(df_ai) == 0:
        raise ValueError(f"No AI samples for generators={generators}")


    if max_ai is not None and len(df_ai) > int(max_ai):
        df_ai = df_ai.sample(int(max_ai), random_state=seed)


    df_bin = pd.concat([df_ai, df_hu], ignore_index=True)

    if shuffle:
        df_bin = df_bin.sample(frac=1.0, random_state=seed).reset_index(drop=True)


    df_train, df_val = train_test_split(
        df_bin,
        test_size=val_size,
        random_state=seed,
        stratify=df_bin[y_col],
    )
    return df_train.reset_index(drop=True), df_val.reset_index(drop=True)


from typing import List
import numpy as np
import pandas as pd


def make_generator_groups(
    df: pd.DataFrame,
    *,
    ai_model_col: str = "model",
    y_col: str = "y",
    min_ai_per_model: int = 2000,
    max_models_per_bert: int = 2,
) -> List[List[str]]:
    """
    Группировка генераторов по max_models_per_bert.
    В группу попадают только генераторы, у которых >= min_ai_per_model AI-примеров.
    """

    df_ai = df[df[y_col] == 1].copy()
    counts = df_ai[ai_model_col].astype(str).value_counts()

    eligible = [m for m, c in counts.items() if int(c) >= min_ai_per_model]
    if not eligible:
        raise ValueError(
            f"No generators have >= {min_ai_per_model} AI samples. "
            f"Lower min_ai_per_model or check ai_model_col."
        )

    groups = []
    for i in range(0, len(eligible), max_models_per_bert):
        groups.append(eligible[i:i + max_models_per_bert])

    return groups


import os
import json


def _print_group_summary(
    df_train: pd.DataFrame,
    df_val: pd.DataFrame,
    *,
    gens: list[str],
    y_col: str,
    ai_model_col: str,
    top_k_models: int = 20,
):
    def _fmt_counts(s: pd.Series, max_items: int = top_k_models) -> str:
        vc = s.value_counts()
        items = []
        for k, v in vc.iloc[:max_items].items():
            items.append(f"{k}: {int(v)}")
        more = ""
        if len(vc) > max_items:
            more = f" | +{len(vc) - max_items} more"
        return ", ".join(items) + more

    def _ratio(d: pd.DataFrame) -> tuple[int, int, float]:
        vc = d[y_col].value_counts()
        n0 = int(vc.get(0, 0))
        n1 = int(vc.get(1, 0))
        r = (n0 / n1) if n1 > 0 else float("inf")
        return n0, n1, r

    def _show_split(name: str, d: pd.DataFrame):
        n0, n1, r = _ratio(d)
        print(f"\n[{name}] rows={len(d)} | n0={n0} n1={n1} | n0/n1={r:.2f}")
        print("  y distribution:")
        print(d[y_col].value_counts(dropna=False).to_string().replace("\n", "\n  "))

        d_ai = d[d[y_col] == 1].copy()
        if len(d_ai) > 0:
            print("  AI model distribution (y==1):")
            print("  " + _fmt_counts(d_ai[ai_model_col].astype(str)))
        else:
            print("  AI model distribution (y==1): <no AI rows>")

    print("\n--- GROUP SUMMARY ---")
    print(f"Generators assigned to this BERT: {gens}")

    for part_name, part_df in [("train", df_train), ("val", df_val)]:
        ai_in = part_df[(part_df[y_col] == 1) & (part_df[ai_model_col].astype(str).isin(gens))]
        ai_out = part_df[(part_df[y_col] == 1) & (~part_df[ai_model_col].astype(str).isin(gens))]
        if len(ai_out) > 0:
            print(f"⚠️  Warning: {part_name} has {len(ai_out)} AI rows OUTSIDE assigned gens (should be 0).")
        print(f"{part_name}: AI rows in assigned gens = {len(ai_in)}")

    _show_split("train", df_train)
    _show_split("val", df_val)
    print("--- END SUMMARY ---\n")


def train_ensemble(
    df: pd.DataFrame,
    *,
    base_model_name: str = "cointegrated/rubert-tiny2",
    text_col: str = "text_for_clf",
    y_col: str = "y",
    ai_model_col: str = "model",
    min_ai_per_model: int = 2000,
    max_models_per_bert: int = 2,
    val_size: float = 0.1,
    seed: int = 40,
    epochs: int = 3,
    batch_size: int = 32,
    max_length: int = 256,
    lr: float = 2e-5,
    weight_decay: float = 0.01,
    ensemble_dir: str = "AIModels",
    print_top_k_models: int = 20,

    max_ai_per_bert: int | None = None,
):
    os.makedirs(ensemble_dir, exist_ok=True)


    groups = make_generator_groups(
        df,
        ai_model_col=ai_model_col,
        y_col=y_col,
        min_ai_per_model=min_ai_per_model,
        max_models_per_bert=max_models_per_bert,
    )


    df_hu_pool = df[df[y_col] == 0].copy()
    df_hu_pool[text_col] = df_hu_pool[text_col].astype(str)
    df_hu_pool = df_hu_pool[df_hu_pool[text_col].str.strip().str.len() > 0]
    n0_all = len(df_hu_pool)

    print("\n=== ENSEMBLE PLAN (generators -> BERT) ===")
    for idx, gens in enumerate(groups, start=1):
        df_ai_pool = df[(df[y_col] == 1) & (df[ai_model_col].astype(str).isin(gens))].copy()
        df_ai_pool[text_col] = df_ai_pool[text_col].astype(str)
        df_ai_pool = df_ai_pool[df_ai_pool[text_col].str.strip().str.len() > 0]
        ai_pool = len(df_ai_pool)
        n1_used = min(ai_pool, max_ai_per_bert) if max_ai_per_bert is not None else ai_pool
        ratio = (n0_all / n1_used) if n1_used > 0 else float("inf")
        print(f"BERT #{idx:02d}: {gens} | ai_pool={ai_pool} | n0_all={n0_all} | n1_used={n1_used} | n0/n1≈{ratio:.2f}")
    print("=== END PLAN ===\n")

    manifest = []
    for idx, gens in enumerate(groups, start=1):
        print(f"\n=== Train BERT #{idx:02d} for generators: {gens} ===")

        df_train, df_val = build_binary_dataset_for_generators(
            df,
            generators=gens,
            text_col=text_col,
            y_col=y_col,
            ai_model_col=ai_model_col,
            val_size=val_size,
            seed=seed + idx,
            max_ai=max_ai_per_bert,
            shuffle=True,
        )

        _print_group_summary(
            df_train,
            df_val,
            gens=gens,
            y_col=y_col,
            ai_model_col=ai_model_col,
            top_k_models=print_top_k_models,
        )

        save_dir = os.path.join(ensemble_dir, f"bert_{idx:02d}")
        os.makedirs(save_dir, exist_ok=True)

        with open(os.path.join(save_dir, "member_config.json"), "w", encoding="utf-8") as f:
            json.dump(
                {
                    "member_id": idx,
                    "generators": gens,
                    "base_model_name": base_model_name,
                    "text_col": text_col,
                    "y_col": y_col,
                    "ai_model_col": ai_model_col,
                    "min_ai_per_model": min_ai_per_model,
                    "max_models_per_bert": max_models_per_bert,
                    "val_size": val_size,
                    "seed": seed + idx,
                    "epochs": epochs,
                    "batch_size": batch_size,
                    "max_length": max_length,
                    "lr": lr,
                    "weight_decay": weight_decay,
                    "max_ai_per_bert": max_ai_per_bert,
                    "negatives_used": "ALL (y==0)",
                },
                f,
                ensure_ascii=False,
                indent=2,
            )


        model, tokenizer, history = train_simple_rubert_tiny2(
            df_train=df_train,
            df_val=df_val,
            text_col=text_col,
            label_col=y_col,
            model_name=base_model_name,
            max_length=max_length,
            batch_size=batch_size,
            lr=lr,
            weight_decay=weight_decay,
            epochs=epochs,
            seed=seed + idx,
            save_dir=save_dir,
        )

        last_epoch_dir = os.path.join(save_dir, f"epoch_{epochs:02d}")
        manifest.append(
            {
                "member_id": idx,
                "generators": gens,
                "model_dir": last_epoch_dir,
            }
        )

    manifest_path = os.path.join(ensemble_dir, "manifest.json")
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print(f"\n Ensemble manifest saved to: {manifest_path}")
    return manifest



In [ ]:
import json
import torch
import numpy as np
from typing import List, Dict, Any
from transformers import AutoTokenizer, AutoModelForSequenceClassification


class EnsembleAIModel:
    def __init__(
        self,
        manifest_path: str,
        *,
        threshold: float = 0.5,
        max_length: int = 256,
        device: str = None,
    ):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        self.threshold = float(threshold)
        self.max_length = int(max_length)

        with open(manifest_path, "r", encoding="utf-8") as f:
            manifest = json.load(f)

        self.members = []
        for m in manifest:
            model_dir = m["model_dir"]
            tok = AutoTokenizer.from_pretrained(model_dir)
            mdl = AutoModelForSequenceClassification.from_pretrained(model_dir)
            mdl.to(self.device)
            mdl.eval()

            self.members.append({
                "tokenizer": tok,
                "model": mdl,
                "meta": m,
            })

    @torch.no_grad()
    def _member_probs_ai(self, member, texts: List[str], batch_size: int = 64) -> np.ndarray:
        tok = member["tokenizer"]
        mdl = member["model"]

        probs = []
        for i in range(0, len(texts), batch_size):
            bt = texts[i:i + batch_size]
            enc = tok(
                bt,
                truncation=True,
                max_length=self.max_length,
                padding=True,
                return_tensors="pt",
            )
            enc = {k: v.to(self.device) for k, v in enc.items()}

            logits = mdl(**enc).logits
            p_ai = torch.softmax(logits, dim=-1)[:, 1]
            probs.append(p_ai.detach().cpu().numpy())

        return np.concatenate(probs)

    def predict_batch(self, texts: List[str], *, batch_size: int = 64) -> List[str]:
        texts = [str(t) for t in texts]
        any_ai = np.zeros(len(texts), dtype=bool)

        for member in self.members:
            p_ai = self._member_probs_ai(member, texts, batch_size=batch_size)
            any_ai |= (p_ai >= self.threshold)

        return ["AI" if x else "not_ai" for x in any_ai]

    def predict(self, text: str) -> str:
        return self.predict_batch([text])[0]

    def predict_proba_or(self, texts: List[str], *, batch_size: int = 64) -> np.ndarray:
        texts = [str(t) for t in texts]
        p_not = np.ones(len(texts), dtype=np.float64)

        for member in self.members:
            p_ai = self._member_probs_ai(member, texts, batch_size=batch_size).astype(np.float64)
            p_not *= (1.0 - p_ai)

        return 1.0 - p_not


    def predict_member_matrix(self, texts: List[str], *, batch_size: int = 64) -> np.ndarray:
        texts = [str(t) for t in texts]
        all_member_probs = []

        for member in self.members:
            p_ai = self._member_probs_ai(member, texts, batch_size=batch_size)
            all_member_probs.append(p_ai)


        return np.vstack(all_member_probs).T


    def predict_generator_group_batch(
        self,
        texts: List[str],
        *,
        batch_size: int = 64,
        threshold: float = None,
    ) -> List[Dict[str, Any]]:
        if threshold is None:
            threshold = self.threshold

        texts = [str(t) for t in texts]
        member_matrix = self.predict_member_matrix(texts, batch_size=batch_size)

        results = []
        for i, text in enumerate(texts):
            probs = member_matrix[i]
            best_idx = int(np.argmax(probs))
            best_prob = float(probs[best_idx])

            best_member = self.members[best_idx]
            member_meta = best_member["meta"]

            results.append({
                "text": text,
                "is_ai_by_best_member": int(best_prob >= threshold),
                "best_member_idx": best_idx,
                "best_member_id": member_meta.get("member_id"),
                "best_member_prob_ai": best_prob,
                "predicted_generators": member_meta.get("generators", []),
                "all_member_probs": probs.tolist(),
            })

        return results


    def predict_generator_group(
        self,
        text: str,
        *,
        threshold: float = None,
    ) -> Dict[str, Any]:
        return self.predict_generator_group_batch([text], threshold=threshold, batch_size=1)[0]


In [ ]:
import numpy as np
import pandas as pd
from typing import Dict, Any
from tqdm.auto import tqdm


def evaluate_generator_routing(
    ens,
    df: pd.DataFrame,
    *,
    text_col: str = "text_for_clf",
    true_model_col: str = "model",
    batch_size: int = 64,
    threshold: float = 0.5,
    verbose: bool = True,
) -> tuple[pd.DataFrame, Dict[str, Any]]:


    df_eval = df.copy()
    df_eval[text_col] = df_eval[text_col].astype(str)
    df_eval[true_model_col] = df_eval[true_model_col].astype(str)

    df_eval = df_eval[df_eval[text_col].str.strip().str.len() > 0].reset_index(drop=True)
    if df_eval.empty:
        raise ValueError("Пустой DataFrame после очистки text_col.")

    texts = df_eval[text_col].tolist()
    true_models = df_eval[true_model_col].tolist()

    all_rows = []

    for start in tqdm(range(0, len(texts), batch_size), desc="routing eval"):
        batch_texts = texts[start:start + batch_size]
        batch_true_models = true_models[start:start + batch_size]

        batch_preds = ens.predict_generator_group_batch(
            batch_texts,
            batch_size=batch_size,
            threshold=threshold,
        )

        for pred, true_model in zip(batch_preds, batch_true_models):
            predicted_generators = pred["predicted_generators"]
            best_prob = float(pred["best_member_prob_ai"])
            passed_threshold = int(best_prob >= threshold)

            hit = int((passed_threshold == 1) and (true_model in predicted_generators))

            row = {
                "true_model": true_model,
                "best_member_id": pred["best_member_id"],
                "best_member_prob_ai": best_prob,
                "passed_threshold": passed_threshold,
                "predicted_generators": predicted_generators,
                "routing_hit": hit,
                "all_member_probs": pred["all_member_probs"],
            }
            all_rows.append(row)

    df_result = pd.concat([df_eval.reset_index(drop=True), pd.DataFrame(all_rows)], axis=1)


    routing_accuracy = float(df_result["routing_hit"].mean())
    coverage = float(df_result["passed_threshold"].mean())


    tp = int(((df_result["passed_threshold"] == 1) & (df_result["routing_hit"] == 1)).sum())
    fp = int(((df_result["passed_threshold"] == 1) & (df_result["routing_hit"] == 0)).sum())
    fn_like = int((df_result["routing_hit"] == 0).sum())

    routing_precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    routing_recall = tp / len(df_result) if len(df_result) > 0 else 0.0

    if routing_precision + routing_recall > 0:
        routing_f1 = 2 * routing_precision * routing_recall / (routing_precision + routing_recall)
    else:
        routing_f1 = 0.0

    by_model = (
        df_result
        .groupby("true_model", dropna=False)["routing_hit"]
        .agg(["count", "mean"])
        .reset_index()
        .rename(columns={"count": "n", "mean": "routing_hit_rate"})
        .sort_values(["routing_hit_rate", "n"], ascending=[False, False])
        .reset_index(drop=True)
    )

    metrics = {
        "n": int(len(df_result)),
        "threshold": float(threshold),
        "routing_accuracy": routing_accuracy,
        "routing_precision": float(routing_precision),
        "routing_recall": float(routing_recall),
        "routing_f1": float(routing_f1),
        "coverage_above_threshold": coverage,
        "mean_best_member_prob_ai": float(df_result["best_member_prob_ai"].mean()),
        "tp": tp,
        "fp": fp,
        "per_model": by_model,
    }

    if verbose:
        print("\n=== ROUTING EVAL ===")
        print(f"n                           : {metrics['n']}")
        print(f"threshold                   : {metrics['threshold']:.4f}")
        print(f"routing_accuracy            : {metrics['routing_accuracy']:.4f}")
        print(f"routing_precision           : {metrics['routing_precision']:.4f}")
        print(f"routing_recall              : {metrics['routing_recall']:.4f}")
        print(f"routing_f1                  : {metrics['routing_f1']:.4f}")
        print(f"coverage_above_threshold    : {metrics['coverage_above_threshold']:.4f}")
        print(f"mean_best_member_prob_ai    : {metrics['mean_best_member_prob_ai']:.4f}")
        print(f"tp                          : {metrics['tp']}")
        print(f"fp                          : {metrics['fp']}")
        print("\n=== PER MODEL ===")
        print(by_model.to_string(index=False))

    return df_result, metrics


In [ ]:
def add_exact_model_score(
    df_result: pd.DataFrame,
    *,
    true_model_col: str = "true_model",
    predicted_generators_col: str = "predicted_generators",
    passed_threshold_col: str = "passed_threshold",
) -> pd.DataFrame:
    df_out = df_result.copy()

    def exact_score(row):
        gens = row[predicted_generators_col]
        true_model = row[true_model_col]
        passed = int(row[passed_threshold_col])

        if passed != 1:
            return 0

        if not isinstance(gens, list):
            return 0

        if len(gens) != 1:
            return 0

        return int(gens[0] == true_model)

    df_out["exact_model_hit"] = df_out.apply(exact_score, axis=1)
    return df_out


In [ ]:
def score_one_text_routing(
    ens,
    text: str,
    true_model: str,
    *,
    threshold: float = 0.5,
) -> Dict[str, Any]:
    pred = ens.predict_generator_group(text, threshold=threshold)

    best_prob = float(pred["best_member_prob_ai"])
    predicted_generators = pred["predicted_generators"]
    passed_threshold = int(best_prob >= threshold)

    routing_hit = int((passed_threshold == 1) and (true_model in predicted_generators))

    exact_model_hit = 0
    if passed_threshold == 1 and len(predicted_generators) == 1:
        exact_model_hit = int(predicted_generators[0] == true_model)

    return {
        "true_model": true_model,
        "best_member_id": pred["best_member_id"],
        "best_member_prob_ai": best_prob,
        "predicted_generators": predicted_generators,
        "passed_threshold": passed_threshold,
        "routing_hit": routing_hit,
        "exact_model_hit": exact_model_hit,
        "all_member_probs": pred["all_member_probs"],
    }


In [ ]:
ens = EnsembleAIModel(
    "ensemble_models_2_3000/manifest.json",
    threshold=0.8,
    max_length=256,
    device="cuda",
)

text = "Твой текст здесь"
true_model = "openai:gpt-4.1-mini"

res_one = score_one_text_routing(
    ens,
    text=text,
    true_model=true_model,
    threshold=0.8,
)

print(res_one)


In [ ]:
import pandas as pd
from typing import Dict, Optional

def filter_val_llmtrace_models(
    df: pd.DataFrame,
    *,
    model_col: str = "model",
    remove_human: bool = True,
    min_count: int = 100,
    family_map: Optional[Dict[str, str]] = None,
    verbose: bool = True,
):

    df = df.copy()
    df[model_col] = df[model_col].astype(str)


    if remove_human:
        df = df[df[model_col].str.lower() != "human"].copy()


    if family_map is None:
        family_map = {

            "gpt-4": "gpt-4",
            "gpt-4-0125-preview": "gpt-4",
            "gpt-4-1106-preview": "gpt-4",

            "gpt-4o": "gpt-4o",

            "gpt-3.5": "gpt-3.5",

            "o1-mini-2024-09-12": "o1",
            "o1-preview-2024-09-12": "o1",
            "o3-2025-04-16": "o3",


            "gigachat": "gigachat",
            "GigaChat-Max": "gigachat",
            "yagpt": "yandexgpt",
            "yandex/YandexGPT-5-Lite-8B-instruct": "yandexgpt",


            "Qwen/QwQ-32B": "qwen_reasoning",
            "Qwen/Qwen2.5-72B-Instruct": "qwen_instruct",
            "Qwen2-7B-Instruct": "qwen_instruct",


            "google/gemma-2-27b-it": "gemma",
            "gemma-1.1-7b-it": "gemma",


            "Meta-Llama-3-8B-Instruct": "llama_instruct",
            "unsloth/Llama-3.3-70B-Instruct": "llama_instruct",
            "llama-7b": "llama_base",


            "Phi-3-mini-128k-instruct": "phi",
            "microsoft/Phi-3-medium-128k-instruct": "phi",


            "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B": "deepseek",
        }

    df["model_family"] = df[model_col].map(lambda x: family_map.get(x, x))


    family_counts = df["model_family"].value_counts()


    keep_families = family_counts[family_counts >= min_count].index.tolist()
    df = df[df["model_family"].isin(keep_families)].copy()


    exact_counts = (
        df.groupby(["model_family", model_col])
        .size()
        .reset_index(name="n")
        .sort_values(["model_family", "n"], ascending=[True, False])
    )

    best_exact_per_family = (
        exact_counts
        .drop_duplicates(subset=["model_family"], keep="first")
        .rename(columns={model_col: "kept_model", "n": "kept_model_count"})
    )

    keep_models = set(best_exact_per_family["kept_model"].tolist())

    df_filtered = df[df[model_col].isin(keep_models)].copy().reset_index(drop=True)

    stats = {
        "family_counts_before_exact_filter": family_counts.to_dict(),
        "best_exact_per_family": best_exact_per_family.reset_index(drop=True),
        "kept_models": sorted(list(keep_models)),
        "n_before": len(df),
        "n_after": len(df_filtered),
    }

    if verbose:
        print("\n=== kept families (count >= min_count) ===")
        print(pd.Series(family_counts[family_counts >= min_count]).sort_values(ascending=False))

        print("\n=== kept exact model for each family ===")
        print(best_exact_per_family.to_string(index=False))

        print("\n=== final exact model counts ===")
        print(df_filtered[model_col].value_counts())

        print(f"\nrows after filtering: {len(df_filtered)}")

    return df_filtered, stats


In [ ]:
df_train_llmtrace['model'].value_counts()


In [ ]:
df_val_llmtrace_filtered, stats = filter_val_llmtrace_models(
    df_val_llmtrace,
    model_col="model",
    remove_human=True,
    min_count=700,
    verbose=True,
)


In [ ]:
df_result, routing_metrics = evaluate_generator_routing(
    ens,
    df=df_val_llmtrace_filtered,
    text_col="text_for_clf",
    true_model_col="model",
    batch_size=64,
    threshold=0.9993,
    verbose=True,
)


In [ ]:

manifest = train_ensemble(
    df_train_llmtrace,
    ai_model_col="model",
    min_ai_per_model=3000,
    max_models_per_bert=2,
    epochs=3,
    ensemble_dir="ensemble_models_2_3000",
)


In [ ]:
ens = EnsembleAIModel(
    "ensemble_models/manifest.json",
    threshold=0.8,
    max_length=256,
)


In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)
from tqdm.auto import tqdm


def evaluate_ensemble_on_df(
    ens,
    df: pd.DataFrame,
    *,
    text_col: str = "text_for_clf",
    label_col: str = "y",
    batch_size: int = 64,
    use_proba_or: bool = False,
    threshold: float = 0.5,
    verbose: bool = True,
    device: str = "cpu",
):

    t_total_start = time.perf_counter()

    df_eval = df[df[label_col].isin([0, 1])].copy()
    df_eval[text_col] = df_eval[text_col].astype(str)
    df_eval = df_eval[df_eval[text_col].str.strip().str.len() > 0].reset_index(drop=True)

    if df_eval.empty:
        raise ValueError("Нет валидных строк для оценки: проверь y и text_for_clf.")

    texts = df_eval[text_col].tolist()
    y_true = df_eval[label_col].astype(int).to_numpy()


    y_pred = np.empty(len(texts), dtype=np.int64)

    infer_time = 0.0

    for i in tqdm(range(0, len(texts), batch_size), desc="ensemble inference"):
        bt = texts[i:i + batch_size]


        if device == "cuda":
            import torch
            torch.cuda.synchronize()

        t_inf = time.perf_counter()

        if use_proba_or:
            p_ai = ens.predict_proba_or(bt, batch_size=batch_size)
            preds = (p_ai >= threshold).astype(np.int64)
        else:
            labels = ens.predict_batch(bt, batch_size=batch_size)
            preds = np.array([1 if x == "AI" else 0 for x in labels], dtype=np.int64)

        if device == "cuda":
            import torch
            torch.cuda.synchronize()

        infer_time += time.perf_counter() - t_inf

        y_pred[i:i + len(bt)] = preds


    acc = accuracy_score(y_true, y_pred)

    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )


    p0, r0, f0, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0], average=None, zero_division=0
    )


    p1, r1, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], average=None, zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    report = classification_report(
        y_true, y_pred,
        labels=[0, 1],
        target_names=["not_ai", "AI"],
        digits=4,
        zero_division=0,
    )

    total_time = time.perf_counter() - t_total_start

    results = {
        "n": int(len(y_true)),
        "accuracy": float(acc),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_macro": float(f_macro),
        "precision_not_ai": float(p0[0]),
        "recall_not_ai": float(r0[0]),
        "f1_not_ai": float(f0[0]),
        "precision_AI": float(p1[0]),
        "recall_AI": float(r1[0]),
        "f1_AI": float(f1[0]),
        "confusion_matrix": cm,
        "classification_report": report,

        "time_total_sec": total_time,
        "time_infer_sec": infer_time,
        "time_per_sample_ms": 1000.0 * infer_time / len(y_true),
    }

    if verbose:
        print("\n=== ENSEMBLE RESULTS ===")
        print(f"n={results['n']}")
        print(f"accuracy        : {results['accuracy']:.4f}")
        print(f"macro P/R/F1    : {results['precision_macro']:.4f} / {results['recall_macro']:.4f} / {results['f1_macro']:.4f}")
        print(f"not_ai P/R/F1   : {results['precision_not_ai']:.4f} / {results['recall_not_ai']:.4f} / {results['f1_not_ai']:.4f}")
        print(f"AI     P/R/F1   : {results['precision_AI']:.4f} / {results['recall_AI']:.4f} / {results['f1_AI']:.4f}")
        print("\nconfusion matrix (true rows, pred cols) [not_ai, AI]:")
        print(results["confusion_matrix"])
        print("\nclassification report:")
        print(results["classification_report"])
        print("\n--- timing ---")
        print(f"total time      : {results['time_total_sec']:.3f} s")
        print(f"inference time  : {results['time_infer_sec']:.3f} s")
        print(f"time / sample   : {results['time_per_sample_ms']:.2f} ms")

    return results


In [ ]:
import time
import numpy as np


def predict_ensemble_one_text(
    ens,
    text: str,
    *,
    use_proba_or: bool = False,
    threshold: float = 0.5,
    device: str = "cpu",
):

    text = str(text)
    if not text.strip():
        raise ValueError("Пустой текст передан в predict_ensemble_one_text.")


    if device == "cuda":
        import torch
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    if use_proba_or:
        p_ai = ens.predict_proba_or([text], batch_size=1)
        p_ai = float(np.asarray(p_ai)[0])
        label = "AI" if p_ai >= threshold else "not_ai"
    else:
        label = ens.predict_batch([text], batch_size=1)[0]
        p_ai = None

    if device == "cuda":
        import torch
        torch.cuda.synchronize()

    infer_time = time.perf_counter() - t0

    return {
        "label": label,
        "p_ai": p_ai,
        "time_infer_sec": infer_time,
        "time_infer_ms": infer_time * 1000.0,
    }


In [ ]:
ens = EnsembleAIModel(
    "ensemble_models/manifest.json",
    threshold=0.8,
    max_length=256,
    device='cpu'
)
res = predict_ensemble_one_text(ens, text)
print(res)


In [ ]:
df_train_all['initial_dataset'].value_counts()
df_ru_news = df_train_all[df_train_all['initial_dataset'] == 'ru_news']
df_ru_news = df_ru_news[df_ru_news['llmUsed'] != 'openai:gpt-4.1-mini']
print(df_ru_news.columns)
df_ru_news['llmUsed'].value_counts()


In [ ]:

ens = EnsembleAIModel(
    "ensemble_models/manifest.json",
    threshold=0.95,
    max_length=300,
    device='cpu'
)


by_gen = evaluate_ensemble_by_generator(
    ens,
    df_val_llmtrace,
    text_col="text_for_clf",
    label_col="y",
    ai_model_col="model",
    batch_size=64,
    min_rows=200,
)
by_gen


In [ ]:



from __future__ import annotations

import os
import re
import json
import time
import threading
from typing import List, Dict, Any, Optional, Tuple, Any as AnyType
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

from tqdm.auto import tqdm

from openai import OpenAI
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type


def _try_import_bertscore():
    try:
        from bert_score import score as bert_score_fn
        return bert_score_fn
    except Exception as e:
        raise RuntimeError(
            "BERTScore is not available. Install it first:\n"
            "  pip install -U bert-score\n"
            f"Import error: {e}"
        )

def _compute_bertscore_batched(
    refs: List[str],
    hyps: List[str],
    *,
    lang: str = "ru",
    model_type: str = "xlm-roberta-large",
    batch_size: int = 32,
    device: Optional[str] = None,
    desc: str = "BERTScore",
) -> Dict[str, Any]:
    if len(refs) != len(hyps):
        raise ValueError(f"refs and hyps must have same length, got {len(refs)} vs {len(hyps)}")

    bert_score_fn = _try_import_bertscore()

    P_all, R_all, F_all = [], [], []
    pbar = tqdm(range(0, len(refs), batch_size), desc=desc, dynamic_ncols=True)

    for i in pbar:
        r = refs[i:i + batch_size]
        h = hyps[i:i + batch_size]

        P, R, F = bert_score_fn(
            cands=h,
            refs=r,
            lang=lang,
            model_type=model_type,
            device=device,
            verbose=False,
            rescale_with_baseline=True,
        )

        P_all.append(P.detach().cpu().numpy())
        R_all.append(R.detach().cpu().numpy())
        F_all.append(F.detach().cpu().numpy())

        f_tmp = np.concatenate(F_all)
        pbar.set_postfix(mean_f1=float(np.mean(f_tmp)), median_f1=float(np.median(f_tmp)))

    P = np.concatenate(P_all).astype(np.float32)
    R = np.concatenate(R_all).astype(np.float32)
    F = np.concatenate(F_all).astype(np.float32)

    return {
        "P": P,
        "R": R,
        "F1": F,
        "mean_F1": float(np.mean(F)),
        "median_F1": float(np.median(F)),
        "p10_F1": float(np.quantile(F, 0.10)),
        "p25_F1": float(np.quantile(F, 0.25)),
        "p75_F1": float(np.quantile(F, 0.75)),
        "p90_F1": float(np.quantile(F, 0.90)),
        "model_type": model_type,
        "lang": lang,
    }


LLM_MAX_CONCURRENCY = int(os.getenv("LLM_MAX_CONCURRENCY", "64"))
OPENAI_MAX_CONCURRENCY = int(os.getenv("OPENAI_MAX_CONCURRENCY", "8"))
DEEPSEEK_MAX_CONCURRENCY = int(os.getenv("DEEPSEEK_MAX_CONCURRENCY", "24"))

_all_sem = threading.Semaphore(max(1, LLM_MAX_CONCURRENCY))
_openai_sem = threading.Semaphore(max(1, OPENAI_MAX_CONCURRENCY))
_deepseek_sem = threading.Semaphore(max(1, DEEPSEEK_MAX_CONCURRENCY))

def _provider_sem(provider: str) -> threading.Semaphore:
    return _openai_sem if provider == "openai" else _deepseek_sem

def make_clients() -> Dict[str, OpenAI]:

    clients: Dict[str, OpenAI] = {}

    openai_key = os.getenv("OPENAI_API_KEY")
    if openai_key:
        clients["openai"] = OpenAI(api_key=openai_key)

    deepseek_key = os.getenv("DEEPSEEK_API_KEY")
    if deepseek_key:
        clients["deepseek"] = OpenAI(api_key=deepseek_key, base_url="https://api.deepseek.com/v1")

    if not clients:
        raise RuntimeError("Не найдены ключи. Установи OPENAI_API_KEY и/или DEEPSEEK_API_KEY.")
    return clients


class TransientRewriteError(Exception):
    pass

class EmptyRewriteError(TransientRewriteError):
    pass

def _is_transient(msg: str) -> bool:
    m = (msg or "").lower()
    return any(x in m for x in ["rate", "timeout", "tempor", "503", "502", "connection", "overload", "try again"])

def _clean(s: str) -> str:
    return (s or "").replace("\x00", "").strip()

_CYR_RE = re.compile(r"[А-Яа-яЁё]")

def _looks_russian(s: str) -> bool:
    return bool(s) and (len(_CYR_RE.findall(s)) >= 3)

def _extract_text_from_responses_api(resp: Any) -> str:
    out = getattr(resp, "output_text", None)
    if out:
        return str(out).strip()

    text = ""
    try:
        for item in getattr(resp, "output", []) or []:
            if getattr(item, "type", None) == "message":
                for c in getattr(item, "content", []) or []:
                    if getattr(c, "type", None) == "output_text":
                        text += getattr(c, "text", "") or ""
    except Exception:
        pass
    return (text or "").strip()


EDIT_OPEN = "<EDIT>"
EDIT_CLOSE = "</EDIT>"
_TAG_BLOCK_RE = re.compile(r"<EDIT>(.*?)</EDIT>", re.DOTALL)

def _extract_edited_from_tagged(fragment_with_tags: str) -> Optional[str]:
    if not fragment_with_tags:
        return None
    m = _TAG_BLOCK_RE.search(fragment_with_tags)
    if not m:
        return None
    inner = (m.group(1) or "").strip()
    return inner if inner else None

SYSTEM_TAG_EDITOR = (
    "Ты — опытный редактор русских текстов. "
    "Тебе пришёл фрагмент, где часть помечена тегами <EDIT>...</EDIT>. "
    "Отредактируй ТОЛЬКО текст внутри <EDIT>...</EDIT>, "
    "не добавляй фактов, сохрани смысл. "
    "Текст ВНЕ тегов НЕ МЕНЯЙ (символ в символ). "
    "Верни весь фрагмент целиком, сохранив теги <EDIT>...</EDIT>."
)

USER_TAG_REWRITE_TEMPLATE = """Отредактируй ТОЛЬКО то, что внутри <EDIT>...</EDIT>.
Правила:
- не добавляй новых фактов
- сохраняй смысл максимально близко
- текст вне тегов НЕ МЕНЯЙ
- верни ВЕСЬ фрагмент целиком, сохранив теги

ФРАГМЕНТ:
{fragment}
"""


@retry(
    wait=wait_exponential(multiplier=1, min=1, max=30),
    stop=stop_after_attempt(6),
    retry=retry_if_exception_type(TransientRewriteError),
)
def _call_llm_rewrite_tagged_fragment(
    *,
    clients: Dict[str, OpenAI],
    fragment_with_tags: str,
    openai_model: str = "gpt-5-mini",
    deepseek_model: str = "deepseek-chat",
    temperature: float = 0.2,
    max_output_tokens: int = 2000,
) -> str:
    frag = _clean(fragment_with_tags)
    if not frag:
        raise EmptyRewriteError("Empty fragment")


    if "openai" in clients:
        with _all_sem:
            with _provider_sem("openai"):
                try:
                    resp = clients["openai"].responses.create(
                        model=openai_model,
                        input=[
                            {"role": "system", "content": [{"type": "input_text", "text": SYSTEM_TAG_EDITOR}]},
                            {"role": "user", "content": [{"type": "input_text", "text": USER_TAG_REWRITE_TEMPLATE.format(fragment=frag)}]},
                        ],
                        max_output_tokens=max_output_tokens,
                    )
                    out = _clean(_extract_text_from_responses_api(resp))
                except Exception as e:
                    msg = str(e)
                    if _is_transient(msg):
                        raise TransientRewriteError(msg)
                    out = ""

        if out and (EDIT_OPEN in out) and (EDIT_CLOSE in out):
            return out


    if "deepseek" not in clients:
        raise EmptyRewriteError("OpenAI empty/bad and DeepSeek unavailable")

    with _all_sem:
        with _provider_sem("deepseek"):
            try:
                resp = clients["deepseek"].chat.completions.create(
                    model=deepseek_model,
                    messages=[
                        {"role": "system", "content": SYSTEM_TAG_EDITOR},
                        {"role": "user", "content": USER_TAG_REWRITE_TEMPLATE.format(fragment=frag)},
                    ],
                    temperature=temperature,
                    max_tokens=max_output_tokens,
                )
                out2 = _clean(resp.choices[0].message.content or "")
                if not out2:
                    raise EmptyRewriteError("DeepSeek returned empty response")
                return out2
            except Exception as e:
                msg = str(e)
                if _is_transient(msg):
                    raise TransientRewriteError(msg)
                raise


class ParallelRewriteClient:
    def __init__(
        self,
        *,
        clients: Dict[str, OpenAI],
        openai_model: str = "gpt-5-mini",
        deepseek_model: str = "deepseek-chat",
        temperature: float = 0.2,
        max_output_tokens: int = 2000,
        max_workers: int = 16,
        verbose_errors: bool = True,
    ):
        self.clients = clients
        self.openai_model = openai_model
        self.deepseek_model = deepseek_model
        self.temperature = float(temperature)
        self.max_output_tokens = int(max_output_tokens)
        self.max_workers = int(max_workers)
        self.verbose_errors = bool(verbose_errors)

    def rewrite_tagged_fragment(self, fragment_with_tags: str) -> str:
        return _call_llm_rewrite_tagged_fragment(
            clients=self.clients,
            fragment_with_tags=fragment_with_tags,
            openai_model=self.openai_model,
            deepseek_model=self.deepseek_model,
            temperature=self.temperature,
            max_output_tokens=self.max_output_tokens,
        )


class ModelAdapter:
    """
    Adapter over your detector/ensemble.
    Required methods on model:
      - predict_proba_ai(texts: List[str], batch_size: int=...) -> array-like shape (n,)
        OR
      - predict_proba_or(...)
    """
    def __init__(self, model: AnyType):
        self.model = model

    def _call_model(self, texts: List[str], *, batch_size: int) -> np.ndarray:
        if hasattr(self.model, "predict_proba_ai") and callable(getattr(self.model, "predict_proba_ai")):
            return np.asarray(self.model.predict_proba_ai(texts, batch_size=batch_size), dtype=np.float32)
        if hasattr(self.model, "predict_proba_or") and callable(getattr(self.model, "predict_proba_or")):
            return np.asarray(self.model.predict_proba_or(texts, batch_size=batch_size), dtype=np.float32)
        raise TypeError("model must implement predict_proba_ai(...) or predict_proba_or(...)")

    def predict_proba_ai_batched(
        self,
        texts: List[str],
        *,
        batch_size: int = 64,
        desc: str = "detector inference",
    ) -> np.ndarray:
        probs = []
        for i in tqdm(range(0, len(texts), batch_size), desc=desc, dynamic_ncols=True):
            bt = texts[i:i + batch_size]
            p = self._call_model(bt, batch_size=batch_size)
            probs.append(np.asarray(p, dtype=np.float32))
        return np.concatenate(probs, axis=0)


def _metrics_from_probs(
    y_true: np.ndarray,
    p_ai: np.ndarray,
    *,
    threshold: float,
    label_names: Tuple[str, str] = ("not_ai", "AI"),
) -> Dict[str, Any]:
    y_pred = (p_ai >= threshold).astype(np.int64)

    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    p0, r0, f0, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0], average=None, zero_division=0)
    p1, r1, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=[1], average=None, zero_division=0)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    report = classification_report(
        y_true, y_pred,
        labels=[0, 1],
        target_names=[label_names[0], label_names[1]],
        digits=4,
        zero_division=0,
    )

    return {
        "n": int(len(y_true)),
        "threshold": float(threshold),
        "accuracy": float(acc),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_macro": float(f_macro),
        "precision_not_ai": float(p0[0]),
        "recall_not_ai": float(r0[0]),
        "f1_not_ai": float(f0[0]),
        "precision_AI": float(p1[0]),
        "recall_AI": float(r1[0]),
        "f1_AI": float(f1[0]),
        "share_pred_AI": float(np.mean(y_pred == 1)),
        "mean_p_ai": float(np.mean(p_ai)),
        "median_p_ai": float(np.median(p_ai)),
        "confusion_matrix": cm,
        "classification_report": report,
    }

def _summary_from_probs(p_ai: np.ndarray, *, threshold: float) -> Dict[str, Any]:
    y_pred = (p_ai >= threshold).astype(np.int64)
    return {
        "n": int(len(p_ai)),
        "threshold": float(threshold),
        "share_pred_AI": float(np.mean(y_pred == 1)),
        "mean_p_ai": float(np.mean(p_ai)),
        "median_p_ai": float(np.median(p_ai)),
        "p10_p_ai": float(np.quantile(p_ai, 0.10)),
        "p90_p_ai": float(np.quantile(p_ai, 0.90)),
    }


_SENT_SPLIT_RE = re.compile(r"[.!?…]+(?:\s+|$)")

def _split_sentences(text: str) -> List[str]:
    parts = _SENT_SPLIT_RE.split(str(text))
    return [p.strip() for p in parts if p and p.strip()]

def _build_context_block(sents: List[str], i: int, *, neighbors: int = 1) -> str:
    n = len(sents)
    left = max(0, i - neighbors)
    right = min(n, i + neighbors + 1)

    parts: List[str] = []
    for j in range(left, right):
        if j == i:
            parts.append(f"{EDIT_OPEN}{sents[j].strip()}{EDIT_CLOSE}")
        else:
            parts.append(sents[j].strip())
    return ". ".join([p for p in parts if p]).strip()

def _score_sentences_p_ai(
    adapter: ModelAdapter,
    sentences: List[str],
    *,
    batch_size: int = 128,
    desc: str = "sent detector",
) -> np.ndarray:
    if not sentences:
        return np.zeros((0,), dtype=np.float32)
    return adapter.predict_proba_ai_batched(sentences, batch_size=batch_size, desc=desc)

def _iterative_rewrite_sentence_until_ok(
    *,
    adapter: ModelAdapter,
    rewrite_client: ParallelRewriteClient,
    sents: List[str],
    sent_i: int,
    sent_threshold: float,
    sent_max_retries: int,
    neighbors: int,
    detector_batch_size: int,
) -> Dict[str, Any]:
    old = sents[sent_i].strip()
    p0 = float(_score_sentences_p_ai(adapter, [old], batch_size=detector_batch_size, desc="sent detector (single)")[0])

    if p0 < float(sent_threshold):
        return {
            "changed": False,
            "sent_id": int(sent_i),
            "old": old,
            "new": old,
            "p_ai_before": p0,
            "p_ai_after": p0,
            "retries": 0,
            "history": [{"try": 0, "p_ai": p0, "text": old}],
        }

    cur = old
    history = [{"try": 0, "p_ai": p0, "text": old}]
    retries_used = 0

    for k in range(1, int(sent_max_retries) + 1):
        tmp_sents = list(sents)
        tmp_sents[sent_i] = cur
        block = _build_context_block(tmp_sents, sent_i, neighbors=neighbors)

        out_block = rewrite_client.rewrite_tagged_fragment(block)
        edited = _extract_edited_from_tagged(out_block)
        edited = _clean(edited or "")

        if not edited:
            history.append({"try": k, "p_ai": None, "text": cur, "note": "empty_or_broken_llm_output"})
            break
        if edited.strip() == cur.strip():
            history.append({"try": k, "p_ai": None, "text": cur, "note": "no_change"})
            break

        p_new = float(_score_sentences_p_ai(adapter, [edited], batch_size=detector_batch_size, desc="sent detector (single)")[0])
        history.append({"try": k, "p_ai": p_new, "text": edited})
        retries_used = k
        cur = edited

        if p_new < float(sent_threshold):
            break

    last = history[-1].get("p_ai", None)
    if last is None:
        p_final = float(_score_sentences_p_ai(adapter, [cur], batch_size=detector_batch_size, desc="sent detector (single)")[0])
        history[-1]["p_ai"] = p_final
    else:
        p_final = float(last)

    return {
        "changed": (cur.strip() != old.strip()),
        "sent_id": int(sent_i),
        "old": old,
        "new": cur,
        "p_ai_before": p0,
        "p_ai_after": p_final,
        "retries": int(retries_used),
        "history": history,
    }


def _load_input_as_df(
    *,
    in_path: str,
    text_col: str,
    label_col: str,
) -> Tuple[pd.DataFrame, bool]:

    path = str(in_path)
    ext = os.path.splitext(path)[1].lower()

    if ext == ".csv":
        df = pd.read_csv(path)
        has_labels = (label_col in df.columns)
        return df, has_labels

    if ext == ".parquet":
        df = pd.read_parquet(path)
        has_labels = (label_col in df.columns)
        return df, has_labels

    if ext in [".jsonl", ".json"]:
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rows.append(json.loads(line))
                except Exception:
                    f.seek(0)
                    obj = json.load(f)
                    if isinstance(obj, list):
                        rows = obj
                    else:
                        rows = [obj]
                    break
        df = pd.DataFrame(rows)
        has_labels = (label_col in df.columns)
        return df, has_labels

    if ext == ".txt":
        texts = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                t = line.strip()
                if t:
                    texts.append(t)
        df = pd.DataFrame({text_col: texts})
        return df, False

    raise ValueError(f"Unsupported file extension: {ext}. Use csv/parquet/jsonl/json/txt")


def audit_detector_before_after(
    *,
    model: AnyType,


    df: Optional[pd.DataFrame] = None,
    in_path: Optional[str] = None,

    text_col: str = "text_for_clf",
    label_col: str = "y",
    threshold: float = 0.5,
    batch_size: int = 64,

    rewrite_client: Optional[ParallelRewriteClient] = None,
    rewrite_policy: str = "quality",
    neighbors: int = 1,
    max_rows: Optional[int] = None,
    sleep_between_calls: float = 0.0,

    compute_bertscore: bool = True,
    bertscore_lang: str = "ru",
    bertscore_model_type: str = "xlm-roberta-large",
    bertscore_batch_size: int = 32,
    bertscore_device: Optional[str] = None,


    save_outputs: bool = False,
    out_rewritten_path: Optional[str] = None,
    out_log_path: Optional[str] = None,


    sent_threshold: float = 0.80,
    sent_max_retries: int = 3,
) -> Dict[str, Any]:
    if (df is None) == (in_path is None):
        raise ValueError("Provide exactly one: df=... OR in_path=...")

    if rewrite_client is None:
        raise RuntimeError("rewrite_client is required for before/after audit.")


    has_labels = True
    if in_path is not None:
        df_in, has_labels = _load_input_as_df(in_path=in_path, text_col=text_col, label_col=label_col)
    else:
        df_in = df.copy()
        has_labels = (label_col in df_in.columns)

    if text_col not in df_in.columns:
        raise ValueError(f"Missing text_col='{text_col}' in input.")

    df_eval = df_in.copy()
    df_eval[text_col] = df_eval[text_col].astype(str)
    df_eval = df_eval[df_eval[text_col].str.strip().str.len() > 0].reset_index(drop=True)

    if has_labels:
        df_eval = df_eval[df_eval[label_col].isin([0, 1])].reset_index(drop=True)

    if max_rows is not None:
        df_eval = df_eval.iloc[: int(max_rows)].reset_index(drop=True)

    if df_eval.empty:
        raise ValueError("No valid rows after filtering.")

    adapter = ModelAdapter(model)

    texts_before = df_eval[text_col].tolist()
    y_true = None
    if has_labels:
        y_true = df_eval[label_col].astype(int).to_numpy()


    p_ai_before = adapter.predict_proba_ai_batched(
        texts_before, batch_size=batch_size, desc="detector inference (before)"
    )
    summary_before = _summary_from_probs(p_ai_before, threshold=threshold)
    metrics_before = _metrics_from_probs(y_true, p_ai_before, threshold=threshold) if has_labels else None


    rewritten_texts: List[str] = []
    rewrite_logs: List[Dict[str, Any]] = []

    n_changed_texts = 0
    n_changed_sents = 0
    n_total_sents = 0
    n_flagged_sents = 0
    n_errors = 0

    pbar = tqdm(
        enumerate(texts_before),
        total=len(texts_before),
        desc=f"rewrite-by-sent-detector (thr={sent_threshold}, retries={sent_max_retries}, neighbors={neighbors})",
        dynamic_ncols=True,
    )

    for row_i, t in pbar:
        try:
            sents = _split_sentences(t)
            n_total_sents += len(sents)

            if not sents:
                rewritten_texts.append(str(t))
                continue


            p_sent = _score_sentences_p_ai(
                adapter, sents,
                batch_size=max(32, int(batch_size)),
                desc="sent detector (select)",
            )

            flagged = [i for i, p in enumerate(p_sent) if float(p) >= float(sent_threshold)]
            n_flagged_sents += len(flagged)

            new_sents = list(sents)
            sent_logs_this_row: List[Dict[str, Any]] = []

            for sent_i in flagged:
                res = _iterative_rewrite_sentence_until_ok(
                    adapter=adapter,
                    rewrite_client=rewrite_client,
                    sents=new_sents,
                    sent_i=sent_i,
                    sent_threshold=sent_threshold,
                    sent_max_retries=sent_max_retries,
                    neighbors=neighbors,
                    detector_batch_size=max(32, int(batch_size)),
                )

                if res["changed"]:
                    new_sents[sent_i] = res["new"]
                    sent_logs_this_row.append(res)

                if sleep_between_calls and sleep_between_calls > 0:
                    time.sleep(float(sleep_between_calls))

            out_text = ". ".join([s.strip() for s in new_sents]).strip()
            if out_text and out_text[-1] not in ".!?…":
                out_text += "."
            rewritten_texts.append(out_text)

            if sent_logs_this_row:
                n_changed_texts += 1
                n_changed_sents += len(sent_logs_this_row)
                for item in sent_logs_this_row:
                    rewrite_logs.append({
                        "row_i": int(row_i),
                        "sent_id": int(item["sent_id"]),
                        "old": item["old"],
                        "new": item["new"],
                        "p_ai_before_sent": float(item["p_ai_before"]),
                        "p_ai_after_sent": float(item["p_ai_after"]),
                        "retries": int(item["retries"]),
                        "history": json.dumps(item["history"], ensure_ascii=False),
                    })

        except Exception as e:
            n_errors += 1
            rewritten_texts.append(t)
            tqdm.write(f"[rewrite error] row={row_i} err={e!r}")

        pbar.set_postfix(
            changed_texts=n_changed_texts,
            changed_sents=n_changed_sents,
            flagged_sents=n_flagged_sents,
            avg_sents_per_text=(n_total_sents / max(1, (row_i + 1))),
            avg_changed_sents_per_changed_text=(n_changed_sents / max(1, n_changed_texts)),
            errors=n_errors,
        )

    df_rewrite_log = pd.DataFrame(rewrite_logs)


    p_ai_after = adapter.predict_proba_ai_batched(
        rewritten_texts, batch_size=batch_size, desc="detector inference (after)"
    )
    summary_after = _summary_from_probs(p_ai_after, threshold=threshold)
    metrics_after = _metrics_from_probs(y_true, p_ai_after, threshold=threshold) if has_labels else None


    bertscore_summary = None
    if compute_bertscore:
        tqdm.write(
            f"\n[BERTScore] computing on {len(texts_before)} pairs | "
            f"lang={bertscore_lang} | model={bertscore_model_type}"
        )
        bs = _compute_bertscore_batched(
            refs=texts_before,
            hyps=rewritten_texts,
            lang=bertscore_lang,
            model_type=bertscore_model_type,
            batch_size=bertscore_batch_size,
            device=bertscore_device,
            desc="BERTScore (orig vs rewrite)",
        )
        bertscore_summary = bs


    print("\n=== BEFORE (detector summary) ===")
    print(f"n={summary_before['n']} | thr={summary_before['threshold']:.3f}")
    print(f"share_pred_AI={summary_before['share_pred_AI']:.3f} | mean_p_ai={summary_before['mean_p_ai']:.3f}")

    print("\n=== AFTER (detector summary) ===")
    print(f"n={summary_after['n']} | thr={summary_after['threshold']:.3f}")
    print(f"share_pred_AI={summary_after['share_pred_AI']:.3f} | mean_p_ai={summary_after['mean_p_ai']:.3f}")

    print("\n=== SHIFT (detector summary) ===")
    print(f"dlt share_pred_AI = {summary_after['share_pred_AI'] - summary_before['share_pred_AI']:+.3f}")
    print(f"dlt mean_p_ai      = {summary_after['mean_p_ai'] - summary_before['mean_p_ai']:+.3f}")

    if has_labels and metrics_before is not None and metrics_after is not None:
        print("\n=== BEFORE (labeled metrics) ===")
        print(f"acc={metrics_before['accuracy']:.4f} | F1_macro={metrics_before['f1_macro']:.4f} | F1_AI={metrics_before['f1_AI']:.4f}")
        print("\n=== AFTER (labeled metrics) ===")
        print(f"acc={metrics_after['accuracy']:.4f} | F1_macro={metrics_after['f1_macro']:.4f} | F1_AI={metrics_after['f1_AI']:.4f}")

    if bertscore_summary is not None:
        f1 = bertscore_summary["F1"]
        print("\n=== SEMANTIC DRIFT (BERTScore orig→rewrite) ===")
        print(
            f"model={bertscore_summary['model_type']} | lang={bertscore_summary['lang']} | "
            f"mean_F1={bertscore_summary['mean_F1']:.4f} | median_F1={bertscore_summary['median_F1']:.4f} | "
            f"p10={bertscore_summary['p10_F1']:.4f} p25={bertscore_summary['p25_F1']:.4f} "
            f"p75={bertscore_summary['p75_F1']:.4f} p90={bertscore_summary['p90_F1']:.4f}"
        )
        print(
            "share(F1<0.80)={:.3f} | share(F1<0.75)={:.3f} | share(F1<0.70)={:.3f}".format(
                float(np.mean(f1 < 0.80)),
                float(np.mean(f1 < 0.75)),
                float(np.mean(f1 < 0.70)),
            )
        )


    if in_path is not None and save_outputs:
        if out_rewritten_path is None:
            out_rewritten_path = os.path.splitext(in_path)[0] + ".rewritten.txt"
        if out_log_path is None:
            out_log_path = os.path.splitext(in_path)[0] + ".rewrite_log.csv"

        with open(out_rewritten_path, "w", encoding="utf-8") as f:
            for t in rewritten_texts:
                f.write(str(t).replace("\n", " ").strip() + "\n")

        df_rewrite_log.to_csv(out_log_path, index=False)

        print(f"\n[saved] rewritten -> {out_rewritten_path}")
        print(f"[saved] log       -> {out_log_path}")

    return {
        "has_labels": bool(has_labels),
        "summary_before": summary_before,
        "summary_after": summary_after,
        "metrics_before": metrics_before,
        "metrics_after": metrics_after,
        "rewrite_log_df": df_rewrite_log,
        "bertscore": bertscore_summary,
        "p_ai_before": p_ai_before,
        "p_ai_after": p_ai_after,
        "rewritten_texts": rewritten_texts if (in_path is not None and save_outputs) else None,
        "sent_threshold": float(sent_threshold),
        "sent_max_retries": int(sent_max_retries),
    }


In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import List, Optional

class HFSequenceClassifierDetector:

    def __init__(
        self,
        model_dir: str,
        *,
        device: Optional[str] = None,
        max_length: int = 256,
    ):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"

        self.device = device
        self.max_length = int(max_length)

        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        self.model.to(self.device)
        self.model.eval()

    @torch.no_grad()
    def predict_proba_ai(self, texts: List[str], batch_size: int = 64) -> np.ndarray:
        texts = [str(t) for t in texts]
        probs = []

        for i in range(0, len(texts), batch_size):
            bt = texts[i:i + batch_size]

            enc = self.tokenizer(
                bt,
                truncation=True,
                max_length=self.max_length,
                padding=True,
                return_tensors="pt",
            )
            enc = {k: v.to(self.device) for k, v in enc.items()}

            logits = self.model(**enc).logits
            p_ai = torch.softmax(logits, dim=-1)[:, 1]
            probs.append(p_ai.detach().cpu().numpy())

        return np.concatenate(probs, axis=0)


In [ ]:
print(len(df_test['rawLabel']))
df_test['rawLabel'].value_counts()


In [ ]:
#obfuscator


from __future__ import annotations

import os
import re
import json
import time
import threading
from typing import List, Tuple, Optional, Dict, Any, Any as AnyType

import numpy as np
from tqdm.auto import tqdm

from openai import OpenAI
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type


LLM_MAX_CONCURRENCY = int(os.getenv("LLM_MAX_CONCURRENCY", "64"))
OPENAI_MAX_CONCURRENCY = int(os.getenv("OPENAI_MAX_CONCURRENCY", "8"))
DEEPSEEK_MAX_CONCURRENCY = int(os.getenv("DEEPSEEK_MAX_CONCURRENCY", "24"))

_all_sem = threading.Semaphore(max(1, LLM_MAX_CONCURRENCY))
_openai_sem = threading.Semaphore(max(1, OPENAI_MAX_CONCURRENCY))
_deepseek_sem = threading.Semaphore(max(1, DEEPSEEK_MAX_CONCURRENCY))

def _provider_sem(provider: str) -> threading.Semaphore:
    return _openai_sem if provider == "openai" else _deepseek_sem

def make_clients() -> Dict[str, OpenAI]:
    clients: Dict[str, OpenAI] = {}

    openai_key = os.getenv("OPENAI_API_KEY")
    if openai_key:
        clients["openai"] = OpenAI(api_key=openai_key)

    deepseek_key = os.getenv("DEEPSEEK_API_KEY") or "sk-b669ef5917f54f20a6f378944872f854"
    if deepseek_key:
        clients["deepseek"] = OpenAI(api_key=deepseek_key, base_url="https://api.deepseek.com/v1")

    if not clients:
        raise RuntimeError("Не найдены ключи. Установи OPENAI_API_KEY и/или DEEPSEEK_API_KEY.")
    return clients


class TransientRewriteError(Exception):
    pass

class EmptyRewriteError(TransientRewriteError):
    pass

def _is_transient(msg: str) -> bool:
    m = (msg or "").lower()
    return any(x in m for x in ["rate", "timeout", "tempor", "503", "502", "connection", "overload", "try again"])

def _clean(s: str) -> str:
    return (s or "").replace("\x00", "").strip()

def _extract_text_from_responses_api(resp: Any) -> str:
    out = getattr(resp, "output_text", None)
    if out:
        return str(out).strip()

    text = ""
    try:
        for item in getattr(resp, "output", []) or []:
            if getattr(item, "type", None) == "message":
                for c in getattr(item, "content", []) or []:
                    if getattr(c, "type", None) == "output_text":
                        text += getattr(c, "text", "") or ""
    except Exception:
        pass
    return (text or "").strip()


EDIT_OPEN = "<EDIT>"
EDIT_CLOSE = "</EDIT>"
_TAG_BLOCK_RE = re.compile(r"<EDIT>(.*?)</EDIT>", re.DOTALL)

def _extract_edited_from_tagged(fragment_with_tags: str) -> Optional[str]:
    if not fragment_with_tags:
        return None
    m = _TAG_BLOCK_RE.search(fragment_with_tags)
    if not m:
        return None
    inner = (m.group(1) or "").strip()
    return inner if inner else None


SYSTEM_TAG_EDITOR = (
    "Ты — опытный редактор русских текстов. "
    "Тебе пришёл фрагмент, где часть помечена тегами <EDIT>...</EDIT>. "
    "Отредактируй ТОЛЬКО текст внутри <EDIT>...</EDIT>, "
    "не добавляй фактов, сохрани смысл. "
    "Текст ВНЕ тегов НЕ МЕНЯЙ (символ в символ). "
    "Верни весь фрагмент целиком, СОХРАНИВ теги <EDIT>...</EDIT>. "
    "Текст внутри тегов должен быть ПЕРЕФРАЗИРОВАН (не допускается идентичный вариант)."
)


USER_TAG_REWRITE_TEMPLATE = """Отредактируй ТОЛЬКО то, что внутри <EDIT>...</EDIT>.
Требования к правке внутри тегов:
- полностью перефразируй (другие слова и конструкции), сохрани смысл и стиль
- не добавляй новых фактов
- избегай дословного совпадения с исходником

ОГРАНИЧЕНИЯ:
- текст ВНЕ тегов <EDIT>...</EDIT> НЕ МЕНЯЙ (символ в символ)
- верни ВЕСЬ фрагмент целиком
- теги <EDIT> и </EDIT> ОБЯЗАТЕЛЬНО должны остаться в ответе

ФРАГМЕНТ:
{fragment}
"""

@retry(
    wait=wait_exponential(multiplier=1, min=1, max=30),
    stop=stop_after_attempt(6),
    retry=retry_if_exception_type(TransientRewriteError),
)
def _call_llm_rewrite_tagged_fragment(
    *,
    clients: Dict[str, OpenAI],
    fragment_with_tags: str,
    openai_model: str = "gpt-5-mini",
    deepseek_model: str = "deepseek-chat",
    temperature: float = 0.2,
    max_output_tokens: int = 2000,
) -> str:
    frag = _clean(fragment_with_tags)
    if not frag:
        raise EmptyRewriteError("Empty fragment")


    if "openai" in clients:
        with _all_sem:
            with _provider_sem("openai"):
                try:
                    resp = clients["openai"].responses.create(
                        model=openai_model,
                        input=[
                            {"role": "system", "content": [{"type": "input_text", "text": SYSTEM_TAG_EDITOR}]},
                            {"role": "user", "content": [{"type": "input_text", "text": USER_TAG_REWRITE_TEMPLATE.format(fragment=frag)}]},
                        ],
                        max_output_tokens=max_output_tokens,
                    )
                    out = _clean(_extract_text_from_responses_api(resp))
                except Exception as e:
                    msg = str(e)
                    if _is_transient(msg):
                        raise TransientRewriteError(msg)
                    out = ""


        if out and (EDIT_OPEN in out) and (EDIT_CLOSE in out):
            return out


    if "deepseek" not in clients:
        raise EmptyRewriteError("OpenAI empty/bad and DeepSeek unavailable")

    with _all_sem:
        with _provider_sem("deepseek"):
            try:
                resp = clients["deepseek"].chat.completions.create(
                    model=deepseek_model,
                    messages=[
                        {"role": "system", "content": SYSTEM_TAG_EDITOR},
                        {"role": "user", "content": USER_TAG_REWRITE_TEMPLATE.format(fragment=frag)},
                    ],
                    temperature=temperature,
                    max_tokens=max_output_tokens,
                )
                out2 = _clean(resp.choices[0].message.content or "")


                if not out2 or (EDIT_OPEN not in out2) or (EDIT_CLOSE not in out2):
                    return ""
                return out2

            except Exception as e:
                msg = str(e)
                if _is_transient(msg):
                    raise TransientRewriteError(msg)
                raise


class ParallelRewriteClient:
    def __init__(
        self,
        *,
        clients: Dict[str, OpenAI],
        openai_model: str = "gpt-5-mini",
        deepseek_model: str = "deepseek-chat",
        temperature: float = 0.2,
        max_output_tokens: int = 2000,
        max_workers: int = 16,
        verbose_errors: bool = True,
    ):
        self.clients = clients
        self.openai_model = openai_model
        self.deepseek_model = deepseek_model
        self.temperature = float(temperature)
        self.max_output_tokens = int(max_output_tokens)
        self.max_workers = int(max_workers)
        self.verbose_errors = bool(verbose_errors)

    def rewrite_tagged_fragment(self, fragment_with_tags: str) -> str:
        return _call_llm_rewrite_tagged_fragment(
            clients=self.clients,
            fragment_with_tags=fragment_with_tags,
            openai_model=self.openai_model,
            deepseek_model=self.deepseek_model,
            temperature=self.temperature,
            max_output_tokens=self.max_output_tokens,
        )


class ModelAdapter:
    def __init__(self, model: AnyType):
        self.model = model

    def _call_model(self, texts: List[str], *, batch_size: int) -> np.ndarray:
        if hasattr(self.model, "predict_proba_ai") and callable(getattr(self.model, "predict_proba_ai")):
            return np.asarray(self.model.predict_proba_ai(texts, batch_size=batch_size), dtype=np.float32)
        if hasattr(self.model, "predict_proba_or") and callable(getattr(self.model, "predict_proba_or")):
            return np.asarray(self.model.predict_proba_or(texts, batch_size=batch_size), dtype=np.float32)
        raise TypeError("model must implement predict_proba_ai(...) or predict_proba_or(...)")

    def predict_proba_ai_batched(
        self,
        texts: List[str],
        *,
        batch_size: int = 64,
        desc: str = "detector inference",
    ) -> np.ndarray:
        probs = []
        for i in tqdm(range(0, len(texts), batch_size), desc=desc, dynamic_ncols=True):
            bt = texts[i:i + batch_size]
            p = self._call_model(bt, batch_size=min(batch_size, len(bt)))
            probs.append(np.asarray(p, dtype=np.float32))
        return np.concatenate(probs, axis=0) if probs else np.zeros((0,), dtype=np.float32)


_SENT_SEG_RE = re.compile(r".*?(?:[.!?…]+(?:\s+|$)|$)", re.DOTALL)

def split_sentences_keep_ws(text: str) -> List[Tuple[str, str]]:
    text = str(text)
    out: List[Tuple[str, str]] = []
    for m in _SENT_SEG_RE.finditer(text):
        seg = m.group(0)
        if not seg:
            continue
        if not seg.strip():
            continue
        core = seg.rstrip()
        trailing_ws = seg[len(core):]
        out.append((core, trailing_ws))
    return out

def join_sentences_keep_ws(parts: List[Tuple[str, str]]) -> str:
    return "".join([core + ws for core, ws in parts])


def build_context_block(cores: List[str], i: int, neighbors: int = 1) -> str:
    left = max(0, i - neighbors)
    right = min(len(cores), i + neighbors + 1)
    chunk: List[str] = []
    for j in range(left, right):
        if j == i:
            chunk.append(f"{EDIT_OPEN}{cores[j]}{EDIT_CLOSE}")
        else:
            chunk.append(cores[j])
    return " ".join([c.strip() for c in chunk if c and c.strip()]).strip()


def score_sentences_p_ai(
    adapter: ModelAdapter,
    sentences: List[str],
    *,
    batch_size: int = 128,
    desc: str = "sent detector",
) -> np.ndarray:
    if not sentences:
        return np.zeros((0,), dtype=np.float32)
    return adapter.predict_proba_ai_batched(sentences, batch_size=batch_size, desc=desc)


def iterative_rewrite_core_until_ok(
    *,
    adapter: ModelAdapter,
    rewrite_client: ParallelRewriteClient,
    cores: List[str],
    i: int,
    neighbors: int,
    sent_threshold: float,
    sent_max_retries: int,
    detector_batch_size: int,
) -> Dict[str, Any]:
    old = cores[i]
    p0 = float(score_sentences_p_ai(adapter, [old], batch_size=detector_batch_size, desc="sent detector (single)")[0])

    history: List[Dict[str, Any]] = [{"try": 0, "p_ai": p0, "text": old}]
    best_text = old
    best_p = p0


    if p0 < float(sent_threshold):
        return {
            "changed": False,
            "sent_id": int(i),
            "old": old,
            "new": old,
            "p_ai_before": p0,
            "p_ai_after": p0,
            "retries": 0,
            "history": history,
        }

    cur = old
    attempts = 0

    for k in range(1, int(sent_max_retries) + 1):
        attempts += 1

        tmp = list(cores)
        tmp[i] = cur
        block = build_context_block(tmp, i, neighbors=neighbors)

        out_block = rewrite_client.rewrite_tagged_fragment(block)
        edited = _clean(_extract_edited_from_tagged(out_block) or "")


        if not edited:
            history.append({"try": k, "p_ai": None, "text": cur, "note": "empty_or_broken_llm_output"})
            continue


        if edited.strip() == cur.strip():
            history.append({"try": k, "p_ai": None, "text": cur, "note": "no_change"})
            continue

        p_new = float(score_sentences_p_ai(adapter, [edited], batch_size=detector_batch_size, desc="sent detector (single)")[0])
        history.append({"try": k, "p_ai": p_new, "text": edited})


        if p_new < best_p:
            best_p = p_new
            best_text = edited


        cur = edited


        if p_new < float(sent_threshold):
            break

    return {
        "changed": (best_text.strip() != old.strip()),
        "sent_id": int(i),
        "old": old,
        "new": best_text,
        "p_ai_before": p0,
        "p_ai_after": float(best_p),
        "retries": int(attempts),
        "history": history,
    }


def make_two_versions_for_file(
    *,
    model: Any,
    rewrite_client: ParallelRewriteClient,
    in_path: str,
    out_marked_path: str,
    out_review_path: str,

    out_clean_path: str,
    sent_threshold: float = 0.80,
    sent_max_retries: int = 3,

    neighbors: int = 1,
    detector_batch_size: int = 256,
    rewrite_sleep: float = 0.0,
    max_lines: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    adapter = ModelAdapter(model)

    with open(in_path, "r", encoding="utf-8") as f:
        lines = [ln.rstrip("\n") for ln in f]

    if max_lines is not None:
        lines = lines[: int(max_lines)]

    marked_lines: List[str] = []
    review_lines: List[str] = []
    clean_lines: List[str] = []

    total_flagged_sents = 0
    total_sents = 0
    total_lines_with_flags = 0
    total_rewritten_sents = 0
    total_errors = 0

    pbar = tqdm(range(len(lines)), desc="file->3 versions", dynamic_ncols=True)

    for idx in pbar:
        text = lines[idx]
        parts = split_sentences_keep_ws(text)
        if not parts:
            marked_lines.append(text)
            review_lines.append(text)
            clean_lines.append(text)
            continue

        cores = [c for c, _ws in parts]
        total_sents += len(cores)


        p_ai = score_sentences_p_ai(
            adapter, cores, batch_size=detector_batch_size, desc="sent detector (select)"
        )

        flagged = [i for i, p in enumerate(p_ai) if float(p) >= float(sent_threshold)]
        if flagged:
            total_lines_with_flags += 1
            total_flagged_sents += len(flagged)

        flagged_set = set(flagged)


        marked_parts: List[Tuple[str, str]] = []
        for i, (core, ws) in enumerate(parts):
            if i in flagged_set:
                marked_parts.append((f"{EDIT_OPEN}{core}{EDIT_CLOSE}", ws))
            else:
                marked_parts.append((core, ws))
        marked_lines.append(join_sentences_keep_ws(marked_parts))


        review_lines.append(text)
        if flagged:
            review_lines.append("")
            review_lines.append("### SUGGESTED EDITS ###")


        clean_cores = list(cores)

        for i in flagged:
            try:
                res = iterative_rewrite_core_until_ok(
                    adapter=adapter,
                    rewrite_client=rewrite_client,
                    cores=clean_cores,
                    i=i,
                    neighbors=neighbors,
                    sent_threshold=sent_threshold,
                    sent_max_retries=sent_max_retries,
                    detector_batch_size=detector_batch_size,
                )

                suggested = (res["new"] or "").strip() or cores[i]

                review_lines.append(
                    f"- sent_id={i} p_ai={float(p_ai[i]):.3f} -> {float(res['p_ai_after']):.3f} | retries={res['retries']}"
                )
                review_lines.append(f"  ORIGINAL: {cores[i]}")
                review_lines.append(f"  SUGGEST:  {suggested}")
                review_lines.append("")

                if res["changed"]:
                    clean_cores[i] = suggested
                    total_rewritten_sents += 1

            except Exception as e:
                total_errors += 1
                if verbose:
                    tqdm.write(f"[rewrite error] line={idx} sent={i} err={e!r}")

                review_lines.append(f"- sent_id={i} p_ai={float(p_ai[i]):.3f} -> (error) | retries=?")
                review_lines.append(f"  ORIGINAL: {cores[i]}")
                review_lines.append(f"  SUGGEST:  {cores[i]}")
                review_lines.append("")


        clean_parts: List[Tuple[str, str]] = []
        for i, (_core, ws) in enumerate(parts):
            clean_parts.append((clean_cores[i], ws))
        clean_lines.append(join_sentences_keep_ws(clean_parts))

        if rewrite_sleep > 0:
            time.sleep(float(rewrite_sleep))

        pbar.set_postfix(
            lines_with_flags=total_lines_with_flags,
            flagged_sents=total_flagged_sents,
            rewritten_sents=total_rewritten_sents,
            total_sents=total_sents,
            share_flagged=(total_flagged_sents / max(1, total_sents)),
            errors=total_errors,
        )

    with open(out_marked_path, "w", encoding="utf-8") as f:
        for ln in marked_lines:
            f.write(ln + "\n")

    with open(out_review_path, "w", encoding="utf-8") as f:
        for ln in review_lines:
            f.write(ln + "\n")

    with open(out_clean_path, "w", encoding="utf-8") as f:
        for ln in clean_lines:
            f.write(ln + "\n")

    stats = {
        "in_path": in_path,
        "out_marked_path": out_marked_path,
        "out_review_path": out_review_path,
        "out_clean_path": out_clean_path,
        "sent_threshold": float(sent_threshold),
        "sent_max_retries": int(sent_max_retries),
        "neighbors": int(neighbors),
        "n_lines": int(len(lines)),
        "n_total_sents": int(total_sents),
        "n_flagged_sents": int(total_flagged_sents),
        "n_rewritten_sents": int(total_rewritten_sents),
        "n_lines_with_flags": int(total_lines_with_flags),
        "share_flagged_sents": float(total_flagged_sents / max(1, total_sents)),
        "errors": int(total_errors),
    }

    if verbose:
        print("\n[done]")
        print(stats)

    return stats


In [ ]:
import numpy as np

class ModelAdapter:

    def __init__(
        self,
        model,
        *,
        tokenizer=None,
        device: str | None = None,
        hf_max_length: int = 256,
        hf_ai_label: int | str = 1,
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.hf_max_length = int(hf_max_length)
        self.hf_ai_label = hf_ai_label


        if device is None:
            try:
                import torch
                device = "cuda" if torch.cuda.is_available() else "cpu"
            except Exception:
                device = "cpu"
        self.device = device


        self._is_hf = hasattr(model, "forward") and hasattr(model, "config") and not (
            hasattr(model, "predict_proba_ai") or hasattr(model, "predict_proba_or")
        )
        if self._is_hf and self.tokenizer is None:
            raise ValueError(
                "HuggingFace model detected, but tokenizer=None. "
                "Create tokenizer and pass it into ModelAdapter(model, tokenizer=...)."
            )


        self._hf_ai_index = None
        if self._is_hf:
            self._hf_ai_index = self._resolve_hf_ai_index(model, hf_ai_label)

    @staticmethod
    def _resolve_hf_ai_index(model, hf_ai_label):

        if isinstance(hf_ai_label, int):
            return int(hf_ai_label)


        s = str(hf_ai_label)
        cfg = getattr(model, "config", None)
        if cfg is None:
            raise ValueError("HF model has no config; cannot resolve hf_ai_label string")

        label2id = getattr(cfg, "label2id", None) or {}
        id2label = getattr(cfg, "id2label", None) or {}


        if s in label2id:
            return int(label2id[s])


        s_low = s.lower()
        for k, v in label2id.items():
            if str(k).lower() == s_low:
                return int(v)
        for idx, name in id2label.items():
            if str(name).lower() == s_low:
                return int(idx)


        num_labels = getattr(cfg, "num_labels", None)
        if num_labels == 2:
            return 1

        raise ValueError(
            f"Cannot resolve hf_ai_label='{hf_ai_label}'. "
            f"Available labels: id2label={id2label}, label2id={label2id}"
        )

    def _call_model_custom(self, texts, *, batch_size: int) -> np.ndarray:
        if hasattr(self.model, "predict_proba_ai") and callable(getattr(self.model, "predict_proba_ai")):
            return np.asarray(self.model.predict_proba_ai(texts, batch_size=batch_size), dtype=np.float32)
        if hasattr(self.model, "predict_proba_or") and callable(getattr(self.model, "predict_proba_or")):
            return np.asarray(self.model.predict_proba_or(texts, batch_size=batch_size), dtype=np.float32)
        raise TypeError("Custom model must implement predict_proba_ai(...) or predict_proba_or(...)")

    def _call_model_hf(self, texts, *, batch_size: int) -> np.ndarray:
        import torch

        self.model.eval()
        probs = []


        for i in range(0, len(texts), batch_size):
            bt = texts[i:i + batch_size]
            enc = self.tokenizer(
                bt,
                padding=True,
                truncation=True,
                max_length=self.hf_max_length,
                return_tensors="pt",
            )
            enc = {k: v.to(self.device) for k, v in enc.items()}

            with torch.no_grad():
                out = self.model(**enc)
                logits = out.logits
                p = torch.softmax(logits, dim=-1)[:, self._hf_ai_index]
                probs.append(p.detach().cpu().numpy().astype(np.float32))

        return np.concatenate(probs, axis=0) if probs else np.zeros((0,), dtype=np.float32)

    def predict_proba_ai_batched(
        self,
        texts,
        *,
        batch_size: int = 64,
        desc: str = "detector inference",
    ) -> np.ndarray:
        from tqdm.auto import tqdm

        texts = ["" if t is None else str(t) for t in texts]
        out = []

        for i in tqdm(range(0, len(texts), batch_size), desc=desc, dynamic_ncols=True):
            bt = texts[i:i + batch_size]
            if self._is_hf:
                p = self._call_model_hf(bt, batch_size=min(batch_size, len(bt)))
            else:
                p = self._call_model_custom(bt, batch_size=min(batch_size, len(bt)))
            out.append(np.asarray(p, dtype=np.float32))

        return np.concatenate(out, axis=0) if out else np.zeros((0,), dtype=np.float32)


In [ ]:
ens = EnsembleAIModel(
    "ensemble_models/manifest.json",
    threshold=0.8,
    max_length=256,
)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda"

tokenizer = AutoTokenizer.from_pretrained("Models/epoch_03")
model = AutoModelForSequenceClassification.from_pretrained("Models/epoch_03").to(device).eval()
adapter = ModelAdapter(model, tokenizer=tokenizer, device=device, hf_max_length=256, hf_ai_label=1)


In [ ]:
clients = make_clients()

rw = ParallelRewriteClient(
    clients=clients,
    openai_model="gpt-5-mini",
    deepseek_model="deepseek-chat",
    max_workers=30,
    max_output_tokens=800,
)

stats = make_two_versions_for_file(
    model=ens,
    rewrite_client=rw,
    in_path="input.txt",
    out_marked_path="input.marked.txt",
    out_review_path="input.review.txt",
    out_clean_path="input.clean.txt",
    sent_threshold=0.80,
    sent_max_retries=3,
    neighbors=0,
)


In [ ]:
res = audit_detector_before_after(
 model=ens,
 in_path="my_data.csv",
 text_col="text_for_clf",
 label_col="y",
 threshold=0.8,
 batch_size=64,
 rewrite_client=rw,
 rewrite_policy="quality",
 neighbors=1,
 max_rows=100,
 compute_bertscore=False,
 save_outputs=True,
 out_rewritten_path="rewritten.txt",
 out_log_path="rewrite_log.csv",
 sent_threshold=0.80,
 sent_max_retries=3,
)


In [ ]:
from model_utils import classify_text, load_model, load_ternary_model
from app import _maybe_enable_cuda_fastpath, _pick_device, _compute_scores_sequential, _hard_cleanup_cuda
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)
import torch


def evaluate_classifier_on_df(
    df: pd.DataFrame,
    *,
    mode: str = "binary",
    text_col: str = "text_for_clf",
    label_col: str = "y",
    label_map: dict | None = None,
    batch_cleanup_every: int = 10,
    device_mode: str = "cpu",
    use_fp16: bool = True,
    min_chars: int = 1,
    verbose: bool = True,
    return_df: bool = True,
):

    _maybe_enable_cuda_fastpath()
    device = _pick_device(device_mode)

    df_eval = df.copy()
    df_eval[text_col] = df_eval[text_col].astype(str)


    df_eval = df_eval[df_eval[text_col].str.strip().str.len() >= int(min_chars)].copy()
    df_eval = df_eval.reset_index(drop=True)

    if df_eval.empty:
        raise ValueError("Нет валидных строк после фильтра по длине текста.")


    y_raw = df_eval[label_col].to_numpy()

    if label_map is not None:
        y_true = np.array([label_map.get(x, -1) for x in y_raw], dtype=np.int64)
    else:
        try:
            y_true = df_eval[label_col].astype(int).to_numpy(dtype=np.int64)
        except Exception:
            raise ValueError(
                "label_col не приводится к int. "
                "Задай label_map, например {'not_ai':0,'AI':1}."
            )

    valid_mask = y_true >= 0
    df_eval = df_eval[valid_mask].reset_index(drop=True)
    y_true = y_true[valid_mask]

    if df_eval.empty:
        raise ValueError("Нет валидных строк после фильтра по label_map/y.")


    if mode == "binary":
        model, scaler, label_encoder, imputer = load_model('./RU_AI_Detector/models/medium_binary_classifier')
        target_names = ["not_ai", "AI"]
        valid_labels = [0, 1]
    else:
        model, scaler, label_encoder, imputer = load_ternary_model()
        target_names = None
        valid_labels = None


    y_pred = np.empty(len(df_eval), dtype=np.int64)
    p_ai = np.full(len(df_eval), np.nan, dtype=np.float32)
    p_human = np.full(len(df_eval), np.nan, dtype=np.float32)

    t_total_start = time.perf_counter()
    infer_time = 0.0

    ctx_autocast = (
        torch.autocast(device_type="cuda", dtype=torch.float16)
        if (device.type == "cuda" and use_fp16)
        else None
    )

    def _predicted_class_to_int(predicted_class: str) -> int:
        pc = str(predicted_class or "").strip().lower()

        if mode == "binary":
            if pc in ("ai", "raw ai", "machine", "llm"):
                return 1
            if pc in ("human", "not_ai", "not ai"):
                return 0
            return 1 if "ai" in pc else 0

        if "human" in pc:
            return 0
        if "reph" in pc or "paraphrase" in pc:
            return 2
        if "ai" in pc:
            return 1
        return -1

    texts = df_eval[text_col].tolist()

    pbar = tqdm(
        enumerate(texts),
        total=len(texts),
        desc="df inference",
    )

    for i, text in pbar:
        if device.type == "cuda":
            torch.cuda.synchronize()

        t_inf = time.perf_counter()

        scores = _compute_scores_sequential(text, device=device)

        with torch.inference_mode():
            if ctx_autocast is not None:
                with ctx_autocast:
                    result = classify_text(
                        text, model, scaler, label_encoder, imputer=imputer, scores=scores
                    )
            else:
                result = classify_text(
                    text, model, scaler, label_encoder, imputer=imputer, scores=scores
                )

        if device.type == "cuda":
            torch.cuda.synchronize()

        dt = time.perf_counter() - t_inf
        infer_time += dt

        predicted_class = (result or {}).get("predicted_class", "Unknown")
        y_pred[i] = _predicted_class_to_int(predicted_class)

        probs = (result or {}).get("probabilities", {}) or {}
        cur_p_ai = None

        for k, v in probs.items():
            kk = str(k).strip().lower()
            try:
                vv = float(v)
            except Exception:
                continue
            if kk in ("ai", "raw ai"):
                p_ai[i] = vv
                cur_p_ai = vv
            if kk in ("human", "not_ai", "not ai"):
                p_human[i] = vv


        avg_ms = (infer_time / (i + 1)) * 1000.0
        postfix = {
            "pred": predicted_class,
            "last_ms": f"{dt * 1000.0:.1f}",
            "avg_ms": f"{avg_ms:.1f}",
        }
        if cur_p_ai is not None:
            postfix["p_ai"] = f"{cur_p_ai:.3f}"

        pbar.set_postfix(postfix)

        if (i + 1) % int(batch_cleanup_every) == 0:
            _hard_cleanup_cuda()

    total_time = time.perf_counter() - t_total_start


    ok_pred_mask = y_pred >= 0
    y_true_m = y_true[ok_pred_mask]
    y_pred_m = y_pred[ok_pred_mask]

    if len(y_true_m) == 0:
        raise ValueError("Все предсказания оказались -1 (Unknown). Проверь маппинг predicted_class.")

    acc = accuracy_score(y_true_m, y_pred_m)

    if mode == "binary":
        labels = [0, 1]
        names = ["not_ai", "AI"]
    else:
        labels = sorted(set(y_true_m.tolist()) | set(y_pred_m.tolist()))
        names = None

    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true_m, y_pred_m, average="macro", zero_division=0
    )
    cm = confusion_matrix(y_true_m, y_pred_m, labels=labels)
    report = classification_report(
        y_true_m,
        y_pred_m,
        labels=labels,
        target_names=names,
        digits=4,
        zero_division=0,
    )

    results = {
        "n_total": int(len(df)),
        "n_used": int(len(df_eval)),
        "n_scored": int(len(y_true_m)),
        "accuracy": float(acc),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_macro": float(f_macro),
        "confusion_matrix": cm,
        "classification_report": report,
        "time_total_sec": float(total_time),
        "time_infer_sec": float(infer_time),
        "time_per_sample_ms": float(1000.0 * infer_time / max(1, len(df_eval))),
        "device": str(device),
        "mode": mode,
    }

    if verbose:
        print("\n=== DF EVAL RESULTS ===")
        print(f"mode={mode} | device={device} | n_used={results['n_used']} | n_scored={results['n_scored']}")
        print(f"accuracy        : {results['accuracy']:.4f}")
        print(f"macro P/R/F1    : {results['precision_macro']:.4f} / {results['recall_macro']:.4f} / {results['f1_macro']:.4f}")
        print("\nconfusion matrix:")
        print(results["confusion_matrix"])
        print("\nclassification report:")
        print(results["classification_report"])
        print("\n--- timing ---")
        print(f"total time      : {results['time_total_sec']:.3f} s")
        print(f"inference time  : {results['time_infer_sec']:.3f} s")
        print(f"time / sample   : {results['time_per_sample_ms']:.2f} ms")

    if return_df:
        df_out = df_eval.copy()
        df_out["y_true"] = y_true
        df_out["y_pred"] = y_pred
        df_out["p_ai"] = p_ai
        df_out["p_human"] = p_human
        return results, df_out

    return results
import os
os.environ["TRANSFORMERS_NO_TQDM"] = "1"
os.environ["TRANSFORMERS_NO_TQDM"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"


In [ ]:
import numpy as np
import pandas as pd

def sample_balanced_binary_df(
    df: pd.DataFrame,
    *,
    text_col: str = "text_for_clf",
    label_col: str = "y",
    n: int = 100,
    min_chars: int = 1000,
    label_map: dict | None = None,
    seed: int = 42,
    keep_only: tuple[int, int] = (0, 1),
) -> pd.DataFrame:

    if n % 2 != 0:
        raise ValueError("n должен быть чётным для баланса 50/50 (например 100).")

    df_eval = df.copy()
    df_eval[text_col] = df_eval[text_col].astype(str)


    df_eval = df_eval[df_eval[text_col].str.strip().str.len() >= int(min_chars)].copy()
    df_eval = df_eval.reset_index(drop=True)

    if df_eval.empty:
        raise ValueError("После фильтра по min_chars не осталось строк.")


    y_raw = df_eval[label_col].to_numpy()

    if label_map is not None:
        y_int = np.array([label_map.get(x, -1) for x in y_raw], dtype=np.int64)
    else:
        try:
            y_int = df_eval[label_col].astype(int).to_numpy(dtype=np.int64)
        except Exception:
            raise ValueError(
                "label_col не приводится к int. "
                "Передай label_map, например {'not_ai':0,'AI':1}."
            )


    keep = set(keep_only)
    mask = np.isin(y_int, list(keep))
    df_eval = df_eval[mask].reset_index(drop=True)
    y_int = y_int[mask]

    if df_eval.empty:
        raise ValueError(f"После фильтра по классам {keep_only} не осталось строк.")


    half = n // 2
    idx0 = np.where(y_int == 0)[0]
    idx1 = np.where(y_int == 1)[0]

    if len(idx0) < half or len(idx1) < half:
        raise ValueError(
            f"Не хватает примеров для 50/50. Нужно минимум {half} каждого класса.\n"
            f"Доступно: y=0 -> {len(idx0)}, y=1 -> {len(idx1)}"
        )

    rng = np.random.default_rng(seed)
    pick0 = rng.choice(idx0, size=half, replace=False)
    pick1 = rng.choice(idx1, size=half, replace=False)
    pick = np.concatenate([pick0, pick1])
    rng.shuffle(pick)

    df_subset = df_eval.iloc[pick].copy().reset_index(drop=True)


    df_subset["_y_int"] = y_int[pick]

    return df_subset


In [ ]:
df_ru_news['classLabel'].value_counts()


In [ ]:
df_100 = sample_balanced_binary_df(
    df_ru_news,
    text_col="text_for_clf",
    label_col="y",
    n=200,
    min_chars=1000,
    seed=123,
)

results, df_pred = evaluate_classifier_on_df(
    df_100,
    mode="binary",
    text_col="text_for_clf",
    label_col="y",
    device_mode="cpu",
    use_fp16=True,
    min_chars=1000,
)


In [ ]:
df.columns


In [ ]:
df_llmtrace.columns


In [ ]:
df_llmtrace['label'].value_counts()


In [ ]:
df['classLabel'].value_counts()


In [ ]:
import pandas as pd
import numpy as np

def normalize_a(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["text"] = df["generatedText"].astype(str)


    cls = df["classLabel"].astype(str).str.strip().str.lower()
    out["class"] = np.where(cls.eq("human"), "Human", "AI")


    out["model"] = df.get("llmUsed", "").astype(str).str.strip()
    out.loc[out["class"].eq("Human"), "model"] = "Human"
    out.loc[out["class"].eq("AI") & (out["model"].eq("") | out["model"].str.lower().eq("nan")), "model"] = "unknown_ai"

    out["source"] = "A"
    return out[["text", "class", "model", "source"]]


def normalize_b(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["text"] = df["text"].astype(str)


    lab = df["label"]
    if pd.api.types.is_numeric_dtype(lab):
        out["class"] = np.where(lab.astype(int) == 1, "AI", "Human")
    else:
        s = lab.astype(str).str.strip().str.lower()
        out["class"] = np.where(s.eq("human"), "Human", "AI")

    out["model"] = df.get("model", "").astype(str).str.strip()
    out.loc[out["class"].eq("Human"), "model"] = "Human"
    out.loc[out["class"].eq("AI") & (out["model"].eq("") | out["model"].str.lower().eq("nan")), "model"] = "unknown_ai"

    out["source"] = "B"
    return out[["text", "class", "model", "source"]]


def merge_two_datasets(df_a: pd.DataFrame, df_b: pd.DataFrame) -> pd.DataFrame:
    a = normalize_a(df_a)
    b = normalize_b(df_b)
    return pd.concat([a, b], ignore_index=True)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import cm

def plot_stacked_by_models_one_bar(
    df_all: pd.DataFrame,
    *,
    min_count_for_separate: int = 1000,
    title: str = "Распределение по моделям генерации",
    out_path: str = "stacked_by_models.png",
    dpi: int = 300,
):

    human_n = int((df_all["class"] == "Human").sum())

    ai_counts = (
        df_all[df_all["class"] == "AI"]
        .groupby("model")
        .size()
        .sort_values(ascending=False)
    )

    big_ai = ai_counts[ai_counts >= min_count_for_separate]
    small_ai = ai_counts[ai_counts < min_count_for_separate]
    other_ai_n = int(small_ai.sum())

    ai_total = int(big_ai.sum() + other_ai_n)


    segments = [("Human", human_n)]
    segments += [(str(m), int(n)) for m, n in big_ai.items()]
    if other_ai_n > 0:
        segments.append(("other_ai", other_ai_n))


    human_color = "#33c4ff"
    ai_keys = [name for name, _ in segments if name != "Human"]
    ai_palette = cm.get_cmap("YlGn", max(3, len(ai_keys)))

    colors = {"Human": human_color}
    for i, k in enumerate(ai_keys):
        colors[k] = ai_palette(0.35 + 0.6 * (i / max(1, len(ai_keys) - 1)))


    fig, ax = plt.subplots(figsize=(14, 6))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    x = ["Категория"]
    bottom = 0

    handles, labels = [], []

    for name, n in segments:
        if n <= 0:
            continue

        bar = ax.bar(
            x,
            [n],
            bottom=[bottom],
            color=colors[name],
            edgecolor="white",
            linewidth=1.0,
            label=name,
        )

        bottom += n
        handles.append(bar[0])
        labels.append(name)


    if human_n > 0:
        ax.text(
            0,
            human_n / 2,
            f"{human_n}",
            ha="center",
            va="center",
            fontsize=13,
            fontweight="bold",
            color="black",
            bbox=dict(
                boxstyle="round,pad=0.2",
                facecolor="white",
                edgecolor="black",
                linewidth=0.9,
                alpha=0.9
            ),
        )


    if ai_total > 0:
        ax.text(
            0,
            human_n + ai_total / 2,
            f"{ai_total}",
            ha="center",
            va="center",
            fontsize=13,
            fontweight="bold",
            color="black",
            bbox=dict(
                boxstyle="round,pad=0.2",
                facecolor="white",
                edgecolor="black",
                linewidth=0.9,
                alpha=0.9
            ),
        )


    ax.set_title(title, fontsize=14)
    ax.set_ylabel("Количество текстов")
    ax.grid(axis="y", linestyle="--", alpha=0.45)

    ax.legend(
        handles,
        labels,
        title="Модели",
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        frameon=False,
    )

    plt.tight_layout()
    plt.savefig(out_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)

    print(f"[saved] {out_path}")


In [ ]:
df.columns


In [ ]:
df_all = merge_two_datasets(df_llmtrace, df)

plot_stacked_by_models_one_bar(
    df_all,
    min_count_for_separate=3000,
    out_path="models_stack_min1000.png"
)


In [ ]:
COLORS = {
    "Human": "#33c4ff",
    "AI":    "#ffd21f",
}
import matplotlib.pyplot as plt
agg = df_all["class"].value_counts()

fig, ax = plt.subplots(figsize=(4, 6))

ax.bar(
    ["Объединённая категория"],
    [agg.get("Human", 0)],
    label="Human",
    color=COLORS["Human"],
)

ax.bar(
    ["Объединённая категория"],
    [agg.get("AI", 0)],
    bottom=[agg.get("Human", 0)],
    label="AI",
    color=COLORS["AI"],
)

ax.set_ylabel("Количество текстов")
ax.legend()
plt.tight_layout()
plt.show()
